## Step 1: Import Libraries and Configuration


In [ ]:
# Import essential libraries and packages for the analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, RobustScaler
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV,
    RandomizedSearchCV, learning_curve
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_regression
import warnings

warnings.filterwarnings('ignore')

# Analysis tools
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import itertools

# Add imports for file downloading
import requests
import os
from pathlib import Path

print("✓ All libraries imported successfully!")


In [ ]:
# Configuration for professional presentation
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style("whitegrid")
sns.set_palette("husl")

# Set personalized random seed
# TODO: Replace XX with the last 2 digits of YOUR student ID
STUDENT_SEED = 18  # Example: Replace with your actual student ID last 2 digits
np.random.seed(STUDENT_SEED)

print(f"✓ Configuration set successfully!")
print(f"Random seed: {STUDENT_SEED}")


# PART 1: ENERGY EFFICIENCY ANALYSIS


## Step 2: Energy Dataset - Download Function


In [ ]:
def download_file(url, local_filename):
    """Download file from URL to local directory if it doesn't exist"""
    if not os.path.exists(local_filename):
        print(f"Downloading {local_filename} from {url}...")
        try:
            response = requests.get(url, stream=True)
            response.raise_for_status()

            with open(local_filename, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"✓ File downloaded successfully: {local_filename}")
            return True
        except Exception as e:
            print(f"✗ Error downloading file: {e}")
            return False
    else:
        print(f"✓ File already exists locally: {local_filename}")
        return True


print("✓ Download function defined")


## Step 3: Energy Dataset - Load Data


In [ ]:
print("=" * 60)
print("PART 1: ENERGY EFFICIENCY ANALYSIS")
print("=" * 60)

# Load Energy Efficiency Dataset with download functionality
print("Loading Energy Efficiency Dataset...")

# Define the URL and local file path
energy_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00242/ENB2012_data.xlsx'
energy_local_file = 'ENB2012_data.xlsx'

# Try to download and read the energy dataset
try:
    download_success = download_file(energy_url, energy_local_file)

    if download_success and os.path.exists(energy_local_file):
        energy_df = pd.read_excel(energy_local_file)
        print(f"✓ Dataset loaded from local file! Shape: {energy_df.shape}")
    else:
        print("Trying direct URL reading as fallback...")
        energy_df = pd.read_excel(energy_url)
        print(f"✓ Dataset loaded from URL! Shape: {energy_df.shape}")

except Exception as e:
    print(f"✗ Error loading dataset: {e}")
    print("Please ensure you have internet connection or the file is available locally")


## Step 4: Energy Dataset - Basic Information


In [ ]:
# Rename columns to descriptive names
energy_df.rename(columns={
    'X1': 'Relative_Compactness',
    'X2': 'Surface_Area',
    'X3': 'Wall_Area',
    'X4': 'Roof_Area',
    'X5': 'Overall_Height',
    'X6': 'Orientation',
    'X7': 'Glazing_Area',
    'X8': 'Glazing_Area_Distribution',
    'Y1': 'Heating_Load',
    'Y2': 'Cooling_Load'
}, inplace=True)

print("Dataset Info:")
print(energy_df.info())
print("\nFirst 5 rows:")
print(energy_df.head())


## Step 5: Energy Dataset - Target Correlation Analysis


In [ ]:
"""
UNDERSTANDING P-VALUE - A DETAILED EXPLANATION:

The p-value is one of the most misunderstood concepts in statistics. Let's break it down:

1. CORRELATION COEFFICIENT (r):
   - Measures LINEAR relationship strength between two variables
   - Range: -1 to +1
   - r = +1: Perfect positive correlation
   - r = 0:  No linear relationship
   - r = -1: Perfect negative correlation
   - |r| > 0.7: Strong correlation
   - 0.3 < |r| < 0.7: Moderate correlation
   - |r| < 0.3: Weak correlation

2. P-VALUE:
   - Probability of observing this correlation (or stronger) by random chance
   - Tests null hypothesis: "No correlation exists (r = 0)"
   - p < 0.05: Statistically significant (reject null hypothesis)
   - p < 0.001: Highly significant
   - p = 0.0000: Extremely significant (actual value < 0.0001)
"""

# Analyze the dual-target nature: Compute correlation between Heating_Load and Cooling_Load
corr, p_value = pearsonr(energy_df['Heating_Load'], energy_df['Cooling_Load'])

print("=" * 80)
print("🔍 DETAILED P-VALUE EXPLANATION WITH OUR DATA")
print("=" * 80)

print(f"📊 Correlation Coefficient (r): {corr:.4f}")
print(f"🎯 P-value: {p_value:.10f}")

print(f"\n🤔 WHAT DOES P-VALUE = {p_value:.10f} MEAN?")
print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

if p_value < 0.0001:
    print(f"🎲 LOTTERY ANALOGY:")
    print(f"   Imagine buying a lottery ticket with 1 in 10,000+ chance of winning")
    print(f"   P-value ≈ 0.0000 means finding this correlation by random chance")
    print(f"   is like winning that lottery - technically possible but extremely unlikely!")

    print(f"\n🔬 SCIENTIFIC INTERPRETATION:")
    print(f"   If we repeated this experiment 10,000 times with truly unrelated variables,")
    print(f"   we'd expect to see correlation ≥ {corr:.4f} in fewer than 1 trial.")

    print(f"\n💡 PLAIN ENGLISH:")
    print(f"   'There's virtually ZERO chance this strong correlation happened by accident.'")
    print(f"   'We can be 99.99%+ confident that heating and cooling loads are truly related.'")

elif p_value < 0.001:
    print(f"   Very strong evidence against random chance")
elif p_value < 0.05:
    print(f"   Strong evidence against random chance")
else:
    print(f"   Weak evidence - could be random")

print(f"\n🎯 NULL HYPOTHESIS TESTING:")
print(f"   Null Hypothesis (H₀): 'Heating and cooling loads are NOT related (r = 0)'")
print(f"   Alternative Hypothesis (H₁): 'Heating and cooling loads ARE related (r ≠ 0)'")
print(f"   ")
print(f"   With p-value ≈ 0.0000:")
print(f"   🚫 REJECT H₀: We have overwhelming evidence to reject the null hypothesis")
print(f"   ✅ ACCEPT H₁: The relationship is statistically significant")

print(f"\n📈 REAL-WORLD IMPLICATIONS:")
print(f"   • This isn't just a number - it has practical meaning!")
print(f"   • We can confidently use one load type to predict the other")
print(f"   • Building designers can focus on factors affecting both loads simultaneously")
print(f"   • The relationship is so strong it's almost like a physical law for this dataset")

# Visual demonstration of what p-value means
print(f"\n🎨 VISUAL ANALOGY:")
print(f"   Imagine plotting 10,000 pairs of truly random, unrelated numbers:")
print(f"   📉📊📈📉📊 (most correlations would be near 0)")
print(f"   🎯 Our correlation of {corr:.4f} would be so extreme it would appear")
print(f"   🌟 less than once in those 10,000 random trials!")

# Sample size consideration
n_samples = len(energy_df)
print(f"\n📊 SAMPLE SIZE CONTEXT:")
print(f"   Sample size (n) = {n_samples}")
print(f"   Larger samples make p-values more reliable")
print(f"   With {n_samples} data points, our p-value calculation is very trustworthy")

# Practical vs Statistical significance
print(f"\n⚖️  STATISTICAL vs PRACTICAL SIGNIFICANCE:")
print(f"   📊 Statistical significance: p ≈ 0.0000 (✅ Very significant)")
print(f"   🏠 Practical significance: r = {corr:.4f} (✅ Very strong relationship)")
print(f"   🎯 Both agree: This is a meaningful, reliable relationship!")

# 4. Check for missing values (df.isnull().sum()), duplicates (df.duplicated().sum()), and data quality issues (e.g., negative values where impossible)

# Data Quality Checks
print("\n--- Data Quality Analysis ---")
print(f"Missing values:\n{energy_df.isnull().sum()}")
print(f"\nDuplicate rows: {energy_df.duplicated().sum()}")
print(f"\nNegative values check:")
for col in energy_df.select_dtypes(include=[np.number]).columns:
    neg_count = (energy_df[col] < 0).sum()
    if neg_count > 0:
        print(f"  {col}: {neg_count} negative values")


## Step 6: Energy Dataset - Heating vs Cooling Load Visualization


In [ ]:
"""
STEP 6: DETAILED LINE-BY-LINE EXPLANATION
=========================================

This step creates two visualizations to examine the relationship between
heating load and cooling load in the energy efficiency dataset.

PURPOSE: Visual analysis helps us understand:
1. How strongly these two variables are related
2. Whether the relationship is linear or non-linear
3. If there are any outliers or unusual patterns
4. The distribution and spread of the data points
"""

# LINE 1: Create a new figure with specified size
plt.figure(figsize=(15, 6))
"""
EXPLANATION:
- plt.figure(): Creates a new matplotlib figure object
- figsize=(15, 6): Sets the figure size to 15 inches wide by 6 inches tall
- This creates the canvas where our plots will be drawn
- Increased width to accommodate two subplots better
"""

# LINE 2: Create the first subplot (left side)
plt.subplot(1, 2, 1)
"""
EXPLANATION:
- plt.subplot(nrows, ncols, index): Divides the figure into a grid of subplots
- (1, 2, 1):
  * 1 row of subplots
  * 2 columns of subplots
  * 1 means this is the 1st subplot (left side)
- This creates a 1×2 grid layout: [Plot1][Plot2]
- We're now working on Plot1 (left side)
"""

# LINE 3: Create a scatter plot
sns.scatterplot(x='Heating_Load', y='Cooling_Load', data=energy_df, alpha=0.6, color='steelblue')
"""
EXPLANATION:
- sns.scatterplot(): Seaborn function to create a scatter plot
- x='Heating_Load': Column name for x-axis values
- y='Cooling_Load': Column name for y-axis values
- data=energy_df: The DataFrame containing our data
- alpha=0.6: Transparency level (0=invisible, 1=opaque)
  * 0.6 makes points semi-transparent so overlapping points are visible
  * Helps identify density patterns where many points cluster
- color='steelblue': Explicitly set point color to steelblue for consistency
"""

# LINE 4: Set title for first subplot
plt.title('Heating Load vs Cooling Load')

# LINE 5: Set x-axis label
plt.xlabel('Heating Load')

# LINE 6: Set y-axis label
plt.ylabel('Cooling Load')

# LINE 7: Create the second subplot (right side)
plt.subplot(1, 2, 2)

# LINE 8: Create a regression plot with explicit blue color
sns.regplot(x='Heating_Load', y='Cooling_Load', data=energy_df,
            scatter_kws={'alpha': 0.6, 'color': 'steelblue'},
            line_kws={'color': 'blue', 'linewidth': 2})
"""
EXPLANATION:
- sns.regplot(): Seaborn function that creates scatter plot + regression line
- x='Heating_Load': Same x-axis variable as before
- y='Cooling_Load': Same y-axis variable as before
- data=energy_df: Same DataFrame
- scatter_kws={'alpha': 0.6, 'color': 'steelblue'}:
  * Dictionary of keyword arguments for scatter points
  * alpha=0.6 makes the points semi-transparent (same as before)
  * color='steelblue' ensures consistent point color with left plot
- line_kws={'color': 'blue', 'linewidth': 2}:
  * Dictionary of keyword arguments for the regression line
  * color='blue' explicitly sets regression line to blue
  * linewidth=2 makes the line slightly thicker for better visibility

WHAT regplot DOES:
1. Creates scatter points (like scatterplot)
2. Fits a linear regression line through the points
3. Adds a confidence interval (shaded area) around the line
4. Shows both the data points AND the trend line in blue
"""

# LINE 9: Set title for second subplot
plt.title('Heating vs Cooling Load (with trend)')
"""
EXPLANATION:
- sets title for the second subplot
- "(with trend)" clarifies that this plot includes the regression line
- Distinguishes it from the first plot which only shows raw data points
"""

# LINE 10: Adjust layout and display
plt.tight_layout()
"""
EXPLANATION:
- plt.tight_layout(): Automatically adjusts subplot spacing
- Prevents overlapping titles, labels, or plots
- Optimizes the use of figure space
- Ensures both subplots fit nicely within the figure boundaries
- Very important when using multiple subplots
"""

# LINE 11: Display the complete figure
plt.show()

print("🎯 WHAT THESE VISUALIZATIONS REVEAL:")
print("=" * 50)
print("Left Plot (Scatter):")
print("  • Shows raw relationship between heating and cooling loads")
print("  • Each point represents one building")
print("  • Transparency helps see overlapping data points")
print("  • Pattern indicates correlation strength")
print("")
print("Right Plot (Regression):")
print("  • Same data with added trend analysis")
print("  • BLUE line = best-fit linear regression")
print("  • Light blue shaded area = confidence interval (uncertainty)")
print("  • Helps quantify the relationship strength")
print("")
print("Expected Observations:")
print(f"  • Strong positive correlation (r ≈ {corr:.3f})")
print("  • Points closely follow the blue trend line")
print("  • Narrow confidence interval indicates reliable relationship")
print("  • Few outliers far from the trend line")
print("")
print("🎨 COLOR SCHEME EXPLANATION:")
print("  • Scatter points: Steel blue (consistent across both plots)")
print("  • Regression line: Bright blue (stands out clearly)")
print("  • This color scheme ensures the trend line is clearly visible")


## Step 7: Energy Dataset - Statistical Summary


In [ ]:
"""
UNDERSTANDING STATISTICAL SUMMARY (describe() function)
======================================================

The describe() function provides essential statistical measures for numerical data.
Each row in the table represents a different statistical metric that helps us
understand the distribution, central tendency, and spread of our data.

Let's break down what each row means:
"""

# Statistical summary
print("--- Statistical Summary ---")
desc_stats = energy_df.describe()
print(desc_stats)

print("\n" + "=" * 80)
print("📊 DETAILED EXPLANATION OF EACH ROW IN THE STATISTICAL SUMMARY")
print("=" * 80)

print("🔢 ROW-BY-ROW EXPLANATION:")
print("─" * 50)

print("1. COUNT:")
print("   • MEANING: Total number of non-missing (valid) data points")
print("   • PURPOSE: Data completeness check")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    count_val = desc_stats.loc['count', col]
    print(f"     - {col}: {count_val} valid observations")
print("   • If count < total rows, we have missing values to handle")

print(f"\n2. MEAN (Average):")
print("   • MEANING: Sum of all values ÷ number of values")
print("   • PURPOSE: Measure of central tendency (typical value)")
print("   • FORMULA: μ = Σx / n")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    mean_val = desc_stats.loc['mean', col]
    print(f"     - {col}: {mean_val:.2f} (typical building uses this much energy)")
print("   • Higher mean suggests higher energy consumption overall")

print(f"\n3. STD (Standard Deviation):")
print("   • MEANING: Average distance of data points from the mean")
print("   • PURPOSE: Measure of variability/spread in the data")
print("   • FORMULA: σ = √(Σ(x - μ)² / n)")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    std_val = desc_stats.loc['std', col]
    mean_val = desc_stats.loc['mean', col]
    cv = (std_val / mean_val) * 100  # Coefficient of variation
    print(f"     - {col}: {std_val:.2f}")
    print(f"       * About 68% of buildings fall within {mean_val - std_val:.1f} to {mean_val + std_val:.1f}")
    print(
        f"       * Coefficient of variation: {cv:.1f}% ({'high' if cv > 30 else 'moderate' if cv > 15 else 'low'} variability)")

print(f"\n4. MIN (Minimum Value):")
print("   • MEANING: Smallest value in the dataset")
print("   • PURPOSE: Lower boundary of the data range")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    min_val = desc_stats.loc['min', col]
    print(f"     - {col}: {min_val:.2f} (most energy-efficient building)")
print("   • Useful for detecting impossible values (e.g., negative energy)")

print(f"\n5. 25% (First Quartile, Q1):")
print("   • MEANING: Value below which 25% of the data falls")
print("   • PURPOSE: Lower middle range boundary")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    q1_val = desc_stats.loc['25%', col]
    print(f"     - {col}: {q1_val:.2f}")
    print(f"       * 25% of buildings use ≤ {q1_val:.1f} units (low energy consumers)")

print(f"\n6. 50% (Median, Q2):")
print("   • MEANING: Middle value when data is sorted (50th percentile)")
print("   • PURPOSE: Robust measure of central tendency")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    median_val = desc_stats.loc['50%', col]
    mean_val = desc_stats.loc['mean', col]
    skew_direction = "right" if mean_val > median_val else "left" if mean_val < median_val else "symmetric"
    print(f"     - {col}: {median_val:.2f} (typical building)")
    print(f"       * Mean vs Median: {mean_val:.1f} vs {median_val:.1f} → {skew_direction} skewed")

print(f"\n7. 75% (Third Quartile, Q3):")
print("   • MEANING: Value below which 75% of the data falls")
print("   • PURPOSE: Upper middle range boundary")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    q3_val = desc_stats.loc['75%', col]
    print(f"     - {col}: {q3_val:.2f}")
    print(f"       * 75% of buildings use ≤ {q3_val:.1f} units")
    print(f"       * 25% of buildings are high energy consumers (> {q3_val:.1f})")

print(f"\n8. MAX (Maximum Value):")
print("   • MEANING: Largest value in the dataset")
print("   • PURPOSE: Upper boundary of the data range")
print("   • INTERPRETATION:")
for col in ['Heating_Load', 'Cooling_Load']:
    max_val = desc_stats.loc['max', col]
    print(f"     - {col}: {max_val:.2f} (least energy-efficient building)")
print("   • Useful for detecting outliers or data entry errors")

print(f"\n🎯 KEY INSIGHTS FROM THE STATISTICS:")
print("=" * 50)

# Calculate additional insights
heating_range = desc_stats.loc['max', 'Heating_Load'] - desc_stats.loc['min', 'Heating_Load']
cooling_range = desc_stats.loc['max', 'Cooling_Load'] - desc_stats.loc['min', 'Cooling_Load']
iqr_heating = desc_stats.loc['75%', 'Heating_Load'] - desc_stats.loc['25%', 'Heating_Load']
iqr_cooling = desc_stats.loc['75%', 'Cooling_Load'] - desc_stats.loc['25%', 'Cooling_Load']

print(f"📏 DATA SPREAD:")
print(
    f"   • Heating Load range: {heating_range:.1f} units (from {desc_stats.loc['min', 'Heating_Load']:.1f} to {desc_stats.loc['max', 'Heating_Load']:.1f})")
print(
    f"   • Cooling Load range: {cooling_range:.1f} units (from {desc_stats.loc['min', 'Cooling_Load']:.1f} to {desc_stats.loc['max', 'Cooling_Load']:.1f})")
print(f"   • Middle 50% (IQR) - Heating: {iqr_heating:.1f}, Cooling: {iqr_cooling:.1f}")

print(f"\n🏠 PRACTICAL IMPLICATIONS:")
print(
    f"   • Average building needs more cooling ({desc_stats.loc['mean', 'Cooling_Load']:.1f}) than heating ({desc_stats.loc['mean', 'Heating_Load']:.1f})")
print(f"   • Energy efficiency varies significantly (7x range from min to max)")
print(f"   • Most buildings cluster in the middle range (median ≈ mean)")
print(f"   • Few extremely high or low energy consumers (relatively normal distribution)")

print(f"\n💡 WHAT THIS TELLS US ABOUT THE DATASET:")
print(f"   ✓ Complete data (no missing values)")
print(f"   ✓ Reasonable value ranges (no negative or impossible values)")
print(f"   ✓ Good variability for machine learning (not all identical)")
print(f"   ✓ Balanced distribution (mean ≈ median suggests no extreme skewness)")


## Step 8: Energy Dataset - Feature Distribution Analysis


In [ ]:
"""
STEP 8: FEATURE DISTRIBUTION ANALYSIS - SIMPLE EXPLANATION
===========================================================

PURPOSE: Create histograms to see how each feature's values are spread out
- Are values normally distributed (bell curve)?
- Are there any unusual patterns or gaps?
- Do we have outliers or extreme values?
"""

# Define which features we want to analyze (exclude target variables)
numerical_features = ['Relative_Compactness', 'Surface_Area', 'Wall_Area',
                      'Roof_Area', 'Overall_Height', 'Glazing_Area']

# Create a grid of subplots (2 rows, 3 columns = 6 plots total)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
"""
EXPLANATION:
- plt.subplots(2, 3): Creates 2 rows and 3 columns of plots
- figsize=(15, 10): Makes the overall figure 15 inches wide, 10 inches tall
- This gives us space for 6 individual histogram plots
"""

# Convert 2D array of axes to 1D for easier indexing
axes = axes.ravel()
"""
EXPLANATION:
- subplots() creates a 2D array: [[ax1, ax2, ax3], [ax4, ax5, ax6]]
- ravel() flattens it to 1D: [ax1, ax2, ax3, ax4, ax5, ax6]
- This makes it easier to loop through with a simple index
"""

# Create a histogram for each feature
for i, feat in enumerate(numerical_features):
    """
    EXPLANATION:
    - enumerate() gives us both index (i) and feature name (feat)
    - i = 0,1,2,3,4,5 for each feature
    - feat = actual feature names like 'Relative_Compactness', etc.
    """

    # Create histogram with density curve overlay
    sns.histplot(energy_df[feat], kde=True, ax=axes[i])
    """
    EXPLANATION:
    - sns.histplot(): Creates a histogram (bar chart showing frequency)
    - energy_df[feat]: Gets the data for this specific feature
    - kde=True: Adds a smooth density curve (Kernel Density Estimation)
    - ax=axes[i]: Puts this plot in the i-th subplot position

    WHAT WE SEE:
    - Bars: How many buildings have each value range
    - Curve: Smooth estimate of the distribution shape
    """

    # Set title for this subplot
    plt.title(f'Distribution of {feat}')
    """
    EXPLANATION:
    - set_title(): Adds a title above each plot
    - f'Distribution of {feat}': Uses f-string to insert feature name
    - Example: "Distribution of Relative_Compactness"
    """

    # Rotate x-axis labels to prevent overlap
    axes[i].tick_params(axis='x', rotation=45)
    """
    EXPLANATION:
    - tick_params(): Adjusts the appearance of axis labels
    - axis='x': Only affect x-axis labels
    - rotation=45: Rotate labels 45 degrees to prevent overlapping
    - This makes long feature names more readable
    """

# Automatically adjust spacing between plots
plt.tight_layout()
"""
EXPLANATION:
- Prevents plots from overlapping each other
- Adjusts margins and spacing automatically
- Makes the overall figure look cleaner and more professional
"""

# Display all the plots
plt.show()
"""
EXPLANATION:
- Actually shows the complete figure with all 6 histograms
- Without this, the plots would be created but not displayed
"""

print("🔍 WHAT TO LOOK FOR IN THESE DISTRIBUTIONS:")
print("=" * 50)
print("✅ GOOD SIGNS:")
print("   • Bell-shaped curves (normal distribution)")
print("   • Smooth, continuous distributions")
print("   • Reasonable value ranges")
print("   • No large gaps in the data")
print("")
print("⚠️  POTENTIAL ISSUES:")
print("   • Highly skewed distributions (long tails)")
print("   • Multiple peaks (bimodal)")
print("   • Extreme outliers")
print("   • Gaps or missing value ranges")
print("")
print("💡 PRACTICAL MEANING:")
print("   • Each histogram shows how building characteristics vary")
print("   • Normal distributions are easier for models to learn")
print("   • Unusual patterns might need special handling")


## Step 9: Energy Dataset - Correlation Matrix


In [ ]:
# Feature correlation analysis
# Analyze relationships between numerical features (-1 to +1 scale)

plt.figure(figsize=(12, 10))  # Large canvas for readability

# Calculate Pearson correlation matrix (measures linear relationships)
# Alternative: .corr(method='spearman') for non-linear monotonic relationships
correlation_matrix = energy_df.corr()

# Create color-coded heatmap
sns.heatmap(
    correlation_matrix,
    annot=True,  # Display correlation values in cells
    cmap='coolwarm',  # Blue=negative, red=positive correlation
    center=0,  # White=no correlation
    square=True,
    linewidths=0.5
)

plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Interpretation guide:
# +1.0: Perfect positive correlation | -1.0: Perfect negative correlation
# 0.0: No linear relationship | >0.7 or <-0.7: Strong correlation (potential multicollinearity)


## Step 10: Energy Dataset - Categorical Features Analysis


In [ ]:
"""
STEP 10: CATEGORICAL FEATURES ANALYSIS
======================================
"""
# ─────────────────────────────────────────────────────────────────────────────
# Goal: Inspect categorical columns and convert them into numeric representations
#       suitable for ML models via one-hot encoding.
# Why: Most scikit-learn estimators accept only numeric arrays; categories must
#      be expanded into indicator (0/1) columns to avoid ordinal assumptions.
# ─────────────────────────────────────────────────────────────────────────────

print("--- Categorical Feature Analysis ---")  # User-facing marker for this step

# Show distinct categories present in 'Orientation' to sanity-check the raw labels
# • Helps confirm the domain (e.g., {2,3,4,5}) and detect unexpected values
print(f"Unique values in Orientation: {energy_df['Orientation'].unique()}")

# Show distinct categories present in 'Glazing_Area_Distribution' for the same reasons
# • Verifies the domain and informs the number of dummy columns that will be created
print(f"Unique values in Glazing_Area_Distribution: {energy_df['Glazing_Area_Distribution'].unique()}")

# Apply one-hot encoding to the two categorical columns:
# • 'columns=[...]' specifies which columns to expand into dummies
# • 'prefix=[...]' controls the column name prefixes to keep features interpretable
# • Output: original columns replaced by multiple binary columns per category value
energy_encoded = pd.get_dummies(
    energy_df,  # Source DataFrame
    columns=['Orientation', 'Glazing_Area_Distribution'],  # Categorical cols
    prefix=['Orient', 'Glaz_Dist']  # Name prefixes for new dummies
)

# Print the post-encoding shape so we see how many columns were added
# (useful to estimate model dimensionality and detect feature explosions)
print(f"Shape after encoding: {energy_encoded.shape}")


## Step 11: Energy Dataset - Outlier Detection


In [ ]:
"""
STEP 11: OUTLIER DETECTION
==========================
"""
# Outlier visualization with boxplots

# Create a grid of 2 rows × 4 columns of subplots.
# plt.subplots returns (figure, axes) where:
#   2, 4               → grid layout (rows, columns)
#   figsize=(16, 8)    → width=16 inches, height=8 inches; increases readability
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# axes is a 2D array [ [ax00, ax01, ...], [ax10, ax11, ...] ].
# .ravel() flattens to 1D (length 8) so we can index with i in a loop.
axes = axes.ravel()

# Features to inspect for outliers—includes both predictors and targets
features_to_check = [
    'Relative_Compactness', 'Surface_Area', 'Wall_Area', 'Roof_Area',
    'Overall_Height', 'Glazing_Area', 'Heating_Load', 'Cooling_Load'
]

for i, feat in enumerate(features_to_check):
    # seaborn.boxplot draws a box-and-whisker chart.
    # params:
    #   y=energy_df[feat] → the data series to plot vertically (box aligns with y-axis)
    #   ax=axes[i]        → explicit Matplotlib Axes object to draw on (the i-th subplot)
    # behavior:
    #   • box shows IQR [Q1,Q3]
    #   • line inside box is median
    #   • whiskers extend to data within 1.5×IQR
    #   • points beyond whiskers are plotted as outliers
    sns.boxplot(y=energy_df[feat], ax=axes[i])

    # Title for this subplot; helps identify which feature is displayed.
    axes[i].set_title(f'Boxplot: {feat}')

# Adjust spacing between subplots to avoid overlapping titles/labels.
# tight_layout() auto-computes paddings based on content.
plt.tight_layout()

plt.show()


## Step 12: Energy Dataset - Feature Engineering


In [ ]:
"""
STEP 12: FEATURE ENGINEERING
============================
"""
# Feature engineering: add domain-inspired signals

print("--- Feature Engineering ---")

# Sum of three area-related columns → a single "total envelope area" indicator.
# Uses vectorized column addition; aligns by index automatically.
energy_encoded['Total_Area'] = (
    energy_encoded['Surface_Area'] +
    energy_encoded['Wall_Area'] +
    energy_encoded['Roof_Area']
)

# A rough proxy for building "volume": surface area × overall height.
# (Not geometric volume but often positively correlated with energy needs.)
energy_encoded['Volume_Proxy'] = (
    energy_encoded['Surface_Area'] *
    energy_encoded['Overall_Height']
)

# Shape–size interaction: relative compactness divided by height.
# Assumes Overall_Height > 0 (true in this dataset).
energy_encoded['Compactness_Height_Ratio'] = (
    energy_encoded['Relative_Compactness'] /
    energy_encoded['Overall_Height']
)

print("✓ Engineered features created:")
print("- Total_Area = Surface_Area + Wall_Area + Roof_Area")
print("- Volume_Proxy = Surface_Area * Overall_Height")
print("- Compactness_Height_Ratio = Relative_Compactness / Overall_Height")


## Step 13: Energy Dataset - Prepare for Modeling


In [ ]:
"""
STEP 13: PREPARE FOR MODELING
=============================
"""
# Train/test preparation (features/target split + holdout)

# Build list of predictor column names by excluding both targets.
# List comprehension iterates over columns and filters out 'Heating_Load' and 'Cooling_Load'.
features_for_model = [
    col for col in energy_encoded.columns
    if col not in ['Heating_Load', 'Cooling_Load']
]

# X: 2D DataFrame of features (n_samples × n_features)
X = energy_encoded[features_for_model]

# y: 1D Series of target values (Heating_Load)
y = energy_encoded['Heating_Load']

print(f"Features for modeling: {len(features_for_model)}")
print(f"Target variable: Heating_Load")

# Split into training and test sets.
# params:
#   test_size=0.2          → 20% of the data becomes the test split
#   random_state=STUDENT_SEED
#                         → makes the split reproducible (same student seed → same split)
#                         → Controls the random number generator used to shuffle the data before splitting
#                         → Same random_state value = identical train/test splits every time you run the code
#                         → Different random_state values = different random splits
#   stratify=None          → not applicable because y is continuous (no class stratification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=STUDENT_SEED, stratify=None
)

# What X_train.shape returns:
# First number: Number of training samples (rows)
# Second number: Number of features (columns)
print(f"Training set shape: {X_train.shape}")  # (n_train, n_features)
print(f"Test set shape: {X_test.shape}")        # (n_test, n_features)


## Step 14: Energy Dataset - Feature Scaling


In [ ]:
"""
STEP 14: FEATURE SCALING
========================
Feature scaling is essential for:
• Gradient-based optimization algorithms (e.g., SGD) to converge faster
• Regularization methods (e.g., Ridge, Lasso) to work effectively
• Ensuring all features contribute equally to distance computations (e.g., in KNN, SVM)
• Improving model interpretability by placing features on a common scale

SCALING STRATEGY:
We use StandardScaler which:
• Centers features by removing the mean
• Scales features to unit variance (dividing by standard deviation)
• Transforms the data to have a standard normal distribution (mean=0, std=1)
"""
print("--- Feature Scaling ---")

# Create a StandardScaler instance.
# It will learn μ (mean) and σ (std) for each feature from the training data only.
scaler = StandardScaler()

# Fit on training features and transform them.
# .fit_transform:
#   • computes μ and σ per column from X_train
#   • applies (x - μ) / σ
X_train_scaled = scaler.fit_transform(X_train)

# Transform test features using the SAME μ and σ learned from training.
# .transform (no .fit!) prevents information leakage.
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled using StandardScaler")
# Quick sanity checks: training-scaled features should have mean≈0, std≈1 per column.
print(f"Training set mean (first 5): {X_train_scaled.mean(axis=0)[:5].round(4)}")
print(f"Training set std (first 5): {X_train_scaled.std(axis=0)[:5].round(4)}")


## Step 15: Energy Dataset - Model Evaluation Function


In [ ]:
"""
STEP 15: MODEL EVALUATION FUNCTION
==================================
"""
# Generic evaluation helper for regression models

def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """Train model, generate predictions, and compute regression metrics."""
    # Fit the model on training data (estimating parameters).
    model.fit(X_train, y_train)

    # Predict target values for the test set (held-out data).
    y_pred = model.predict(X_test)

    # Error metrics:
    # mean_squared_error(y_true, y_pred) → average of (error)^2
    mse = mean_squared_error(y_test, y_pred)

    # Root of MSE to bring error back to target units (same units as y).
    rmse = np.sqrt(mse)

    # mean_absolute_error(y_true, y_pred) → average absolute deviation
    mae = mean_absolute_error(y_test, y_pred)

    # r2_score(y_true, y_pred) → proportion of variance explained (can be negative)
    r2 = r2_score(y_test, y_pred)

    return {
        'model': model,          # return the fitted estimator for later use (e.g., coefficients, predict)
        'predictions': y_pred,   # cached predictions to avoid recomputing
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2': r2
    }

print("✓ Model evaluation function defined")


## Step 16: Energy Dataset - Train Linear Regression


In [ ]:
"""
STEP 16: TRAIN LINEAR REGRESSION
================================
"""
# Baseline: Linear Regression (OLS)

print("--- Model Training and Evaluation ---")
energy_results = {}  # dict to store metrics by model name

print("Training Linear Regression...")
lr_model = LinearRegression()  # Ordinary Least Squares; no regularization

# Use the scaled features for numerically stable coefficients/solver steps.
energy_results['Linear'] = evaluate_model(
    lr_model,
    X_train_scaled,   # scaled training X
    X_test_scaled,    # scaled test X
    y_train,          # unscaled y (target usually not scaled for interpretability)
    y_test,
    'Linear Regression'
)

# Report two key metrics: R² (higher is better) and RMSE (lower is better).
print(f"✓ Linear Regression - R²: {energy_results['Linear']['r2']:.4f}, "
      f"RMSE: {energy_results['Linear']['rmse']:.4f}")


## Step 17: Energy Dataset - Train Polynomial Regression


In [ ]:
"""
STEP 17: TRAIN POLYNOMIAL REGRESSION
====================================
"""
# Polynomial Regression via Pipeline (degrees 2–4)

polynomial_degrees = [2, 3, 4]

for degree in polynomial_degrees:
    print(f"Training Polynomial Regression (degree {degree})...")

    # sklearn.pipeline.Pipeline chains multiple steps into one estimator.
    # Steps:
    #   ('poly', PolynomialFeatures(...)) → expands columns with polynomial terms
    #   ('scaler', StandardScaler())      → standardizes the expanded feature space
    #   ('reg', LinearRegression())       → OLS in the transformed space
    #
    # PolynomialFeatures params:
    #   degree=degree       → maximum polynomial degree
    #   include_bias=False  → do NOT add a constant 1 column (the regressor adds intercept)
    poly_pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ])

    # Evaluate with raw X (not pre-scaled) because the pipeline contains its own scaler.
    energy_results[f'Poly_{degree}'] = evaluate_model(
        poly_pipeline,
        X_train, X_test, y_train, y_test,
        f'Polynomial {degree}'
    )

    print(f"✓ Polynomial {degree} - R²: {energy_results[f'Poly_{degree}']['r2']:.4f}, "
          f"RMSE: {energy_results[f'Poly_{degree}']['rmse']:.4f}")


## Step 18: Energy Dataset - Results Summary


In [ ]:
"""
STEP 18: RESULTS SUMMARY
========================
"""
# Summarize model performances and pick best by R²

print("--- Energy Efficiency Results ---")
# Nice fixed-width header: column labels with left alignment and width specs
print(f"{'Model':<20} {'R²':<8} {'RMSE':<8} {'MAE':<8} {'MSE':<10}")
print("-" * 60)

# Iterate dict items; format floats to 4 decimals for readability
for model_name, results in energy_results.items():
    print(f"{model_name:<20} {results['r2']:<8.4f} {results['rmse']:<8.4f} "
          f"{results['mae']:<8.4f} {results['mse']:<10.4f}")

# max(..., key=lambda ...) selects the key (model name) with the largest R²
energy_best = max(energy_results.keys(), key=lambda x: energy_results[x]['r2'])
print(f"\n🏆 Best Energy Model: {energy_best} with R² = {energy_results[energy_best]['r2']:.4f}")


## Step 19: Energy Dataset - Feature Importance Analysis


In [ ]:
"""
STEP 19: FEATURE IMPORTANCE ANALYSIS
====================================
"""
# Linear-model feature importance (coefficients on scaled features)

print("--- Feature Importance Analysis ---")

# Build a DataFrame mapping each feature to its learned coefficient.
# energy_results['Linear']['model'] is the fitted LinearRegression on X_train_scaled.
feature_importance = pd.DataFrame({
    'feature': features_for_model,
    'coefficient': energy_results['Linear']['model'].coef_
})

# Absolute value allows ranking by magnitude regardless of positive/negative sign.
feature_importance['abs_coefficient'] = np.abs(feature_importance['coefficient'])

# Sort descending by |coef| so most influential features appear first.
feature_importance = feature_importance.sort_values('abs_coefficient', ascending=False)

print(f"Top 5 most important features:")
for i, row in feature_importance.head().iterrows():
    print(f"  {row['feature']}: {row['coefficient']:.4f}")


## Step 20: Energy Dataset - Visualization


In [ ]:
"""
STEP 20: VISUALIZATION
=====================
"""
# Visualizations: feature importance (bar) and predicted vs actual (scatter)

plt.figure(figsize=(12, 8))  # overall canvas for both subplots

# Subplot 1: top-10 absolute coefficients
plt.subplot(1, 2, 1)
top_features = feature_importance.head(10)

# seaborn.barplot draws a bar chart.
# params (common):
#   data=top_features          → DataFrame providing columns
#   x='abs_coefficient'        → numeric column on X-axis (bar length)
#   y='feature'                → categorical column on Y-axis (bar labels)
sns.barplot(data=top_features, x='abs_coefficient', y='feature')

plt.title('Top 10 Feature Importance (Linear Regression)')  # readable title
plt.xlabel('Absolute Coefficient Value')                    # axis label (units: scaled)

# Subplot 2: predicted vs actual for the chosen best model
plt.subplot(1, 2, 2)

# Retrieve cached predictions from the best model we picked earlier.
best_predictions = energy_results[energy_best]['predictions']

# Matplotlib scatter:
#   x=y_test                   → true target values on x-axis
#   y=best_predictions         → predicted target values on y-axis
#   alpha=0.6                  → semi-transparency so dense areas show density
plt.scatter(y_test, best_predictions, alpha=0.6)

# Add a 45° reference line for "perfect predictions".
#   [y_test.min(), y_test.max()] → x-coordinates spanning the data range
#   [y_test.min(), y_test.max()] → y-coordinates equal to x (identity line)
#   'r--'                        → red dashed line style
#   lw=2                         → line width 2 points
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)

plt.xlabel('Actual Heating Load')
plt.ylabel('Predicted Heating Load')
plt.title(f'Predicted vs Actual ({energy_best})')

plt.tight_layout()  # adjust spacing to avoid overlap
plt.show()


## Step 29: Energy Dataset - Advanced Preprocessing and Feature Engineering


In [ ]:
"""
STEP 29: ADVANCED PREPROCESSING AND FEATURE ENGINEERING
=======================================================
REQUIREMENT ALIGNMENT: Addresses requirements 1-3 - Scale features, engineer features, split data
"""
print("="*60)
print("PART 2: ENHANCED ENERGY EFFICIENCY ANALYSIS")
print("="*60)

# Start with the energy_encoded dataset from Part 1
print("--- Advanced Feature Engineering ---")

# Create additional engineered features as required
print("Creating engineered features:")

# Feature Engineering Requirements:
# 1. Total_Area = Surface_Area + Wall_Area + Roof_Area
energy_encoded['Total_Area'] = (
    energy_encoded['Surface_Area'] +
    energy_encoded['Wall_Area'] +
    energy_encoded['Roof_Area']
)
print("✓ Total_Area = Surface_Area + Wall_Area + Roof_Area")

# 2. Volume = Surface_Area * Overall_Height
energy_encoded['Volume'] = (
    energy_encoded['Surface_Area'] *
    energy_encoded['Overall_Height']
)
print("✓ Volume = Surface_Area * Overall_Height")

# Additional meaningful engineered features
# 3. Compactness per unit volume
energy_encoded['Compactness_Volume_Ratio'] = (
    energy_encoded['Relative_Compactness'] /
    (energy_encoded['Volume'] + 1e-8)  # Avoid division by zero
)
print("✓ Compactness_Volume_Ratio = Relative_Compactness / Volume")

# 4. Glazing efficiency ratio
energy_encoded['Glazing_Efficiency'] = (
    energy_encoded['Glazing_Area'] /
    (energy_encoded['Total_Area'] + 1e-8)
)
print("✓ Glazing_Efficiency = Glazing_Area / Total_Area")

# 5. Wall to roof ratio
energy_encoded['Wall_Roof_Ratio'] = (
    energy_encoded['Wall_Area'] /
    (energy_encoded['Roof_Area'] + 1e-8)
)
print("✓ Wall_Roof_Ratio = Wall_Area / Roof_Area")

print(f"\nFinal dataset shape after feature engineering: {energy_encoded.shape}")


## Step 30: Energy Dataset - Data Splitting and Scaling


In [ ]:
"""
STEP 30: DATA SPLITTING AND SCALING
===================================
REQUIREMENT ALIGNMENT: Addresses requirement 3 - Split data with specified parameters
"""
print("--- Data Splitting and Scaling ---")

# Prepare features and target
features_for_enhanced_model = [col for col in energy_encoded.columns
                              if col not in ['Heating_Load', 'Cooling_Load']]

X_energy = energy_encoded[features_for_enhanced_model]
y_energy_heating = energy_encoded['Heating_Load']
y_energy_cooling = energy_encoded['Cooling_Load']

print(f"Features for modeling: {len(features_for_enhanced_model)}")
print(f"Feature names: {features_for_enhanced_model}")

# Split data as required: test_size=0.2, random_state=STUDENT_SEED
X_energy_train, X_energy_test, y_heating_train, y_heating_test = train_test_split(
    X_energy, y_energy_heating, test_size=0.2, random_state=STUDENT_SEED
)

print(f"Training set shape: {X_energy_train.shape}")
print(f"Test set shape: {X_energy_test.shape}")

# Scale numerical features using StandardScaler (fit on train set)
print("\nApplying StandardScaler...")
energy_scaler = StandardScaler()

# Fit scaler on training data only to prevent data leakage
X_energy_train_scaled = energy_scaler.fit_transform(X_energy_train)
X_energy_test_scaled = energy_scaler.transform(X_energy_test)

print("✓ Features scaled using StandardScaler (fitted on training set)")
print(f"Training set mean (first 5): {X_energy_train_scaled.mean(axis=0)[:5].round(4)}")
print(f"Training set std (first 5): {X_energy_train_scaled.std(axis=0)[:5].round(4)}")


## Step 31: Energy Dataset - Target Distribution Analysis


In [ ]:
"""
STEP 31: TARGET DISTRIBUTION ANALYSIS
=====================================
REQUIREMENT ALIGNMENT: Addresses requirement 4 - Check for class balance in targets
"""
print("--- Target Distribution Analysis ---")

# Analyze Heating Load distribution
print("📊 HEATING LOAD DISTRIBUTION:")
print(f"   Mean: {y_energy_heating.mean():.3f}")
print(f"   Std: {y_energy_heating.std():.3f}")
print(f"   Min: {y_energy_heating.min():.3f}")
print(f"   Max: {y_energy_heating.max():.3f}")
print(f"   Range: {y_energy_heating.max() - y_energy_heating.min():.3f}")

# Create heating load bins for balance analysis (optional discretization for insight)
heating_bins = pd.qcut(y_energy_heating, q=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
heating_distribution = heating_bins.value_counts().sort_index()

print(f"\n📊 HEATING LOAD QUINTILE DISTRIBUTION:")
for category, count in heating_distribution.items():
    percentage = count / len(y_energy_heating) * 100
    print(f"   {category}: {count} samples ({percentage:.1f}%)")

# Visualize target distributions
plt.figure(figsize=(15, 6))

plt.subplot(1, 3, 1)
plt.hist(y_energy_heating, bins=30, alpha=0.7, color='red', edgecolor='black')
plt.title('Heating Load Distribution')
plt.xlabel('Heating Load')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.hist(y_energy_cooling, bins=30, alpha=0.7, color='blue', edgecolor='black')
plt.title('Cooling Load Distribution')
plt.xlabel('Cooling Load')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
heating_distribution.plot(kind='bar', color='orange', alpha=0.7)
plt.title('Heating Load Quintile Balance')
plt.xlabel('Load Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Balance analysis conclusion
cv_heating = heating_distribution.std() / heating_distribution.mean()
print(f"\n💡 TARGET BALANCE ASSESSMENT:")
print(f"   Coefficient of Variation: {cv_heating:.3f}")
if cv_heating < 0.2:
    print(f"   ✅ Well-balanced target distribution")
else:
    print(f"   ⚠️ Some imbalance in target distribution")


## Step 32: Energy Dataset - Model Implementation and Evaluation


In [ ]:
"""
STEP 32: MODEL IMPLEMENTATION AND EVALUATION
============================================
REQUIREMENT ALIGNMENT: Addresses requirements 1-7 - Implement models and evaluate
"""
print("--- Enhanced Model Implementation ---")

# Initialize results dictionary
enhanced_energy_results = {}

# 1. Baseline: LinearRegression()
print("1. Training Baseline Linear Regression...")
lr_baseline = LinearRegression()
enhanced_energy_results['Linear_Baseline'] = evaluate_model(
    lr_baseline, X_energy_train_scaled, X_energy_test_scaled,
    y_heating_train, y_heating_test, 'Linear Regression Baseline'
)

print(f"✓ Linear Baseline - R²: {enhanced_energy_results['Linear_Baseline']['r2']:.4f}, "
      f"RMSE: {enhanced_energy_results['Linear_Baseline']['rmse']:.4f}")

# 2. Polynomial Models: degrees 2, 3, 4
polynomial_degrees = [2, 3, 4]

for degree in polynomial_degrees:
    print(f"2.{degree}. Training Polynomial Regression (degree {degree})...")

    # Pipeline with PolynomialFeatures
    poly_pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ])

    # Use unscaled data as pipeline includes its own scaler
    enhanced_energy_results[f'Polynomial_{degree}'] = evaluate_model(
        poly_pipeline, X_energy_train, X_energy_test,
        y_heating_train, y_heating_test, f'Polynomial {degree}'
    )

    print(f"✓ Polynomial {degree} - R²: {enhanced_energy_results[f'Polynomial_{degree}']['r2']:.4f}, "
          f"RMSE: {enhanced_energy_results[f'Polynomial_{degree}']['rmse']:.4f}")

# 3. Comprehensive Model Evaluation
print(f"\n--- Comprehensive Model Evaluation ---")
print(f"{'Model':<20} {'R²':<10} {'RMSE':<10} {'MAE':<10} {'MSE':<12}")
print("─" * 70)

for model_name, results in enhanced_energy_results.items():
    print(f"{model_name:<20} {results['r2']:<10.4f} {results['rmse']:<10.4f} "
          f"{results['mae']:<10.4f} {results['mse']:<12.4f}")

# Find best model
best_energy_model = max(enhanced_energy_results.keys(),
                       key=lambda x: enhanced_energy_results[x]['r2'])
print(f"\n🏆 Best Model: {best_energy_model} with R² = {enhanced_energy_results[best_energy_model]['r2']:.4f}")


## Step 33: Energy Dataset - Model Complexity Analysis (R² Differences)


In [ ]:
"""
STEP 33: MODEL COMPLEXITY ANALYSIS
==================================
REQUIREMENT ALIGNMENT: Addresses requirement 4 - What does difference in R² tell us about model complexity?
"""
print("="*80)
print("🎯 REQUIREMENT 4: R² DIFFERENCES AND MODEL COMPLEXITY ANALYSIS")
print("="*80)

baseline_r2 = enhanced_energy_results['Linear_Baseline']['r2']
print(f"📊 R² PROGRESSION ANALYSIS:")
print(f"   Linear Baseline R² = {baseline_r2:.6f}")

complexity_analysis = []
for degree in polynomial_degrees:
    model_key = f'Polynomial_{degree}'
    if model_key in enhanced_energy_results:
        poly_r2 = enhanced_energy_results[model_key]['r2']
        improvement = poly_r2 - baseline_r2
        percent_improvement = (improvement / baseline_r2) * 100

        complexity_analysis.append({
            'degree': degree,
            'r2': poly_r2,
            'improvement': improvement,
            'percent_improvement': percent_improvement
        })

        print(f"   Polynomial Degree {degree}:")
        print(f"      R² = {poly_r2:.6f}")
        print(f"      Improvement = +{improvement:.6f} ({percent_improvement:+.2f}%)")

        # Interpret the improvement
        if improvement > 0.05:
            interpretation = "🚀 MAJOR improvement - significant non-linear patterns captured"
        elif improvement > 0.01:
            interpretation = "✅ SIGNIFICANT improvement - meaningful complexity benefit"
        elif improvement > 0.001:
            interpretation = "📈 MODERATE improvement - some non-linear relationships"
        elif improvement > 0:
            interpretation = "📊 MINOR improvement - limited benefit from complexity"
        else:
            interpretation = "⚠️ NO improvement - potential overfitting or noise"

        print(f"      → {interpretation}")

print(f"\n💡 MODEL COMPLEXITY INSIGHTS:")
print(f"• R² measures proportion of variance explained (0 to 1)")
print(f"• Higher polynomial degrees add complexity (more parameters)")
print(f"• Diminishing returns indicate optimal complexity threshold")
print(f"• Large R² jumps suggest important non-linear relationships")

# Calculate marginal improvements
if len(complexity_analysis) > 1:
    print(f"\n📈 MARGINAL COMPLEXITY BENEFITS:")
    for i in range(1, len(complexity_analysis)):
        current = complexity_analysis[i]
        previous = complexity_analysis[i-1]
        marginal_gain = current['r2'] - previous['r2']
        print(f"   Degree {previous['degree']} → {current['degree']}: +{marginal_gain:.6f} R²")

        if marginal_gain < 0.001:
            print(f"   💡 Diminishing returns detected at degree {current['degree']}")
            break


## Step 34: Energy Dataset - MSE Performance Analysis


In [ ]:
"""
STEP 34: MSE PERFORMANCE ANALYSIS
=================================
REQUIREMENT ALIGNMENT: Addresses requirement 5 - What does difference in MSE tell us about performance?
"""
print("="*80)
print("🎯 REQUIREMENT 5: MSE DIFFERENCES AND PERFORMANCE ANALYSIS")
print("="*80)

baseline_mse = enhanced_energy_results['Linear_Baseline']['mse']
baseline_rmse = enhanced_energy_results['Linear_Baseline']['rmse']

print(f"📉 MSE/RMSE PROGRESSION ANALYSIS:")
print(f"   Linear Baseline: MSE = {baseline_mse:.6f} | RMSE = {baseline_rmse:.4f}")

for degree in polynomial_degrees:
    model_key = f'Polynomial_{degree}'
    if model_key in enhanced_energy_results:
        poly_mse = enhanced_energy_results[model_key]['mse']
        poly_rmse = enhanced_energy_results[model_key]['rmse']
        mse_reduction = baseline_mse - poly_mse
        rmse_reduction = baseline_rmse - poly_rmse
        percent_reduction = (mse_reduction / baseline_mse) * 100

        print(f"   Polynomial Degree {degree}:")
        print(f"      MSE = {poly_mse:.6f} | RMSE = {poly_rmse:.4f}")
        print(f"      MSE reduction = {mse_reduction:.6f} ({percent_reduction:+.2f}%)")
        print(f"      RMSE reduction = {rmse_reduction:.4f}")

print(f"\n🏠 HEATING LOAD MSE INTERPRETATION:")
print(f"• MSE = Mean Squared Error (squared units: heating load²)")
print(f"• RMSE = Root MSE (same units as Heating_Load)")
print(f"• Current baseline RMSE ≈ {baseline_rmse:.2f} means average prediction error is ~{baseline_rmse:.2f} units")

print(f"\n⚠️ WHAT HIGH MSE IN HEATING LOAD PREDICTION MEANS:")
print(f"• BUILDING DESIGN IMPACT: Poor predictions lead to:")
print(f"  - Incorrect HVAC system sizing")
print(f"  - Over/under-estimated energy costs")
print(f"  - Inefficient building operation")
print(f"  - Suboptimal insulation planning")
print(f"• PRACTICAL CONSEQUENCES:")
print(f"  - Economic waste from oversized heating systems")
print(f"  - Comfort issues from undersized systems")
print(f"  - Environmental impact from energy inefficiency")
print(f"  - Regulatory compliance problems")

print(f"\n📊 MSE REDUCTION BENEFITS:")
best_mse_model = min(enhanced_energy_results.keys(),
                    key=lambda x: enhanced_energy_results[x]['mse'])
best_mse = enhanced_energy_results[best_mse_model]['mse']
mse_improvement = baseline_mse - best_mse
print(f"• Best model ({best_mse_model}) reduces MSE by {mse_improvement:.6f}")
print(f"• This translates to {np.sqrt(mse_improvement):.3f} units better RMSE")
print(f"• Practical impact: More accurate energy predictions for building design")


## Step 35: Energy Dataset - Visualization


In [ ]:
"""
STEP 35: VISUALIZATION
=====================
REQUIREMENT ALIGNMENT: Addresses requirement 6 - Scatter predicted vs actual
"""
print("--- Prediction Visualization ---")

# Create comprehensive visualization
plt.figure(figsize=(16, 12))

# Plot 1: Predicted vs Actual for all models
model_names = list(enhanced_energy_results.keys())
n_models = len(model_names)

for i, model_name in enumerate(model_names):
    plt.subplot(2, 3, i+1)

    predictions = enhanced_energy_results[model_name]['predictions']

    # Scatter plot
    plt.scatter(y_heating_test, predictions, alpha=0.6, s=20)

    # Perfect prediction line
    min_val = min(y_heating_test.min(), predictions.min())
    max_val = max(y_heating_test.max(), predictions.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

    # Add model metrics
    r2 = enhanced_energy_results[model_name]['r2']
    rmse = enhanced_energy_results[model_name]['rmse']

    plt.xlabel('Actual Heating Load')
    plt.ylabel('Predicted Heating Load')
    plt.title(f'{model_name}\nR² = {r2:.4f}, RMSE = {rmse:.4f}')
    plt.legend()
    plt.grid(True, alpha=0.3)

# Residual analysis for best model
if len(model_names) >= 5:
    plt.subplot(2, 3, 6)
    best_predictions = enhanced_energy_results[best_energy_model]['predictions']
    residuals = y_heating_test - best_predictions

    plt.scatter(best_predictions, residuals, alpha=0.6, s=20)
    plt.axhline(y=0, color='r', linestyle='--')
    plt.xlabel('Predicted Heating Load')
    plt.ylabel('Residuals')
    plt.title(f'Residual Analysis - {best_energy_model}')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Step 36: Energy Dataset - Feature Importance Analysis


In [ ]:
"""
STEP 36: FEATURE IMPORTANCE ANALYSIS
====================================
REQUIREMENT ALIGNMENT: Addresses requirement 7 - Feature importance for LR, plot coefficients
"""
print("--- Feature Importance Analysis ---")

# Get feature importance from linear regression
linear_model = enhanced_energy_results['Linear_Baseline']['model']
feature_names = features_for_enhanced_model

# Create feature importance DataFrame
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': linear_model.coef_,
    'abs_coefficient': np.abs(linear_model.coef_)
})

feature_importance_df = feature_importance_df.sort_values('abs_coefficient', ascending=False)

print(f"🏆 TOP 10 MOST IMPORTANT FEATURES:")
print(f"{'Rank':<5} {'Feature':<25} {'Coefficient':<15} {'Abs. Coefficient':<15}")
print("─" * 70)

for i, (_, row) in enumerate(feature_importance_df.head(10).iterrows(), 1):
    print(f"{i:<5} {row['feature']:<25} {row['coefficient']:<+15.6f} {row['abs_coefficient']:<15.6f}")

# Visualization
plt.figure(figsize=(15, 10))

# Plot 1: Top 15 feature coefficients
plt.subplot(2, 2, 1)
top_15_features = feature_importance_df.head(15)
colors = ['red' if x < 0 else 'blue' for x in top_15_features['coefficient']]
plt.barh(range(len(top_15_features)), top_15_features['coefficient'], color=colors, alpha=0.7)
plt.yticks(range(len(top_15_features)), top_15_features['feature'])
plt.xlabel('Coefficient Value')
plt.title('Top 15 Feature Coefficients (Scaled)')
plt.grid(True, alpha=0.3)

# Plot 2: Feature importance by category
plt.subplot(2, 2, 2)
# Categorize features
categories = {
    'Original': ['Relative_Compactness', 'Surface_Area', 'Wall_Area', 'Roof_Area', 'Overall_Height', 'Glazing_Area'],
    'Engineered': ['Total_Area', 'Volume', 'Compactness_Volume_Ratio', 'Glazing_Efficiency', 'Wall_Roof_Ratio'],
    'Categorical': [col for col in feature_names if col.startswith(('Orient_', 'Glaz_Dist_'))]
}

category_importance = {}
for category, features in categories.items():
    category_features = [f for f in features if f in feature_names]
    if category_features:
        avg_importance = feature_importance_df[
            feature_importance_df['feature'].isin(category_features)
        ]['abs_coefficient'].mean()
        category_importance[category] = avg_importance

plt.bar(category_importance.keys(), category_importance.values(), alpha=0.7)
plt.ylabel('Average |Coefficient|')
plt.title('Feature Importance by Category')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

# Plot 3: Coefficient distribution
plt.subplot(2, 2, 3)
plt.hist(feature_importance_df['coefficient'], bins=20, alpha=0.7, edgecolor='black')
plt.xlabel('Coefficient Value')
plt.ylabel('Frequency')
plt.title('Distribution of Feature Coefficients')
plt.grid(True, alpha=0.3)

# Plot 4: Original vs Engineered feature importance
plt.subplot(2, 2, 4)
original_features = feature_importance_df[
    feature_importance_df['feature'].isin(categories['Original'])
]
engineered_features = feature_importance_df[
    feature_importance_df['feature'].isin(categories['Engineered'])
]

if len(original_features) > 0 and len(engineered_features) > 0:
    plt.boxplot([original_features['abs_coefficient'].values,
                 engineered_features['abs_coefficient'].values],
                labels=['Original', 'Engineered'])
    plt.ylabel('|Coefficient|')
    plt.title('Original vs Engineered Feature Importance')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 FEATURE IMPORTANCE INSIGHTS:")
top_feature = feature_importance_df.iloc[0]
print(f"• MOST IMPORTANT: {top_feature['feature']} (coef = {top_feature['coefficient']:+.6f})")

if top_feature['feature'] in categories['Engineered']:
    print(f"  → Engineered feature shows highest predictive power")
else:
    print(f"  → Original building characteristic dominates")

print(f"• POSITIVE COEFFICIENTS: Increase heating load")
print(f"• NEGATIVE COEFFICIENTS: Decrease heating load (improve efficiency)")


## Step 37: Energy Dataset - Heating vs Cooling Load Relationship


In [ ]:
"""
STEP 37: HEATING VS COOLING LOAD RELATIONSHIP
=============================================
REQUIREMENT ALIGNMENT: Addresses requirement 8 - Relationship between heating and cooling loads
"""
print("="*80)
print("🎯 REQUIREMENT 8: HEATING VS COOLING LOAD RELATIONSHIP")
print("="*80)

# Comprehensive correlation analysis
heating_cooling_corr, heating_cooling_p = pearsonr(y_energy_heating, y_energy_cooling)
spearman_corr, spearman_p = spearmanr(y_energy_heating, y_energy_cooling)

print(f"📊 HEATING-COOLING LOAD CORRELATION ANALYSIS:")
print(f"   Pearson correlation:  r = {heating_cooling_corr:.6f} (p = {heating_cooling_p:.2e})")
print(f"   Spearman correlation: ρ = {spearman_corr:.6f} (p = {spearman_p:.2e})")

# Fit linear relationship
heating_cooling_model = LinearRegression()
heating_cooling_model.fit(y_energy_heating.values.reshape(-1, 1), y_energy_cooling)
slope = heating_cooling_model.coef_[0]
intercept = heating_cooling_model.intercept_
r2_relationship = heating_cooling_model.score(y_energy_heating.values.reshape(-1, 1), y_energy_cooling)

print(f"\n📈 LINEAR RELATIONSHIP MODEL:")
print(f"   Cooling_Load = {slope:.6f} × Heating_Load + {intercept:.6f}")
print(f"   R² = {r2_relationship:.6f} ({r2_relationship*100:.2f}% variance explained)")

# Interpretation
print(f"\n🏠 PRACTICAL IMPLICATIONS:")
if abs(heating_cooling_corr) > 0.8:
    strength = "EXTREMELY STRONG"
elif abs(heating_cooling_corr) > 0.6:
    strength = "STRONG"
elif abs(heating_cooling_corr) > 0.4:
    strength = "MODERATE"
else:
    strength = "WEAK"

print(f"• {strength} correlation (r = {heating_cooling_corr:.3f})")
print(f"• Buildings with high heating needs {'also' if heating_cooling_corr > 0 else 'do NOT'} have high cooling needs")

if heating_cooling_corr > 0:
    print(f"• COMMON UNDERLYING FACTORS affecting both loads:")
    print(f"  ✓ Building envelope efficiency (insulation quality)")
    print(f"  ✓ Window properties and solar heat gain")
    print(f"  ✓ Building geometry and thermal mass")
    print(f"  ✓ Air leakage and ventilation characteristics")
    print(f"• ECONOMIC IMPACT: Slope {slope:.3f} means 1 unit ↑ heating → {slope:.3f} units ↑ cooling cost")

# Detailed visualization
plt.figure(figsize=(16, 10))

plt.subplot(2, 2, 1)
plt.scatter(energy_df['Heating_Load'], energy_df['Cooling_Load'], alpha=0.6, color='steelblue')
plt.plot(energy_df['Heating_Load'],
         slope * energy_df['Heating_Load'] + intercept,
         'r-', linewidth=2, label=f'y = {slope:.2f}x + {intercept:.2f}')
plt.xlabel('Heating Load')
plt.ylabel('Cooling Load')
plt.title(f'Heating vs Cooling Load (r = {pearson_r:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)

# Residual analysis for relationship quality assessment
plt.subplot(2, 2, 2)
residuals = energy_df['Cooling_Load'] - (slope * energy_df['Heating_Load'] + intercept)
plt.scatter(energy_df['Heating_Load'], residuals, alpha=0.6, color='orange')
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Heating Load')
plt.ylabel('Residuals')
plt.title('Residual Analysis')
plt.grid(True, alpha=0.3)

# Additional subplots for comprehensive analysis
plt.subplot(2, 2, 3)
plt.hist(residuals, bins=30, alpha=0.7, color='green', edgecolor='black')
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Residual Distribution')
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 4)
from scipy import stats
stats.probplot(residuals, dist="norm", plot=plt)
plt.title('Q-Q Plot: Normality Check')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Step 38: Building Feature Correlation Analysis - ENHANCED


In [ ]:
"""
STEP 38: COMPREHENSIVE FEATURE CORRELATION WITH ENERGY LOADS
============================================================
REQUIREMENT ALIGNMENT: Enhanced version of requirement 9
"""

print("="*80)
print("🎯 REQUIREMENT 9: STRONGEST FEATURE CORRELATIONS WITH ENERGY LOADS")
print("="*80)

# Comprehensive correlation analysis for all numeric features
numeric_features = ['Relative_Compactness', 'Surface_Area', 'Wall_Area', 'Roof_Area',
                   'Overall_Height', 'Glazing_Area']

print("📊 DETAILED CORRELATION ANALYSIS:")
print(f"{'Feature':<25} {'Heating_Load':<15} {'Cooling_Load':<15} {'Avg_Abs':<10} {'Strength':<12}")
print("─" * 85)

correlation_analysis = []
for feature in numeric_features:
    heating_corr = energy_df[feature].corr(energy_df['Heating_Load'])
    cooling_corr = energy_df[feature].corr(energy_df['Cooling_Load'])
    avg_abs_corr = (abs(heating_corr) + abs(cooling_corr)) / 2

    # Classify correlation strength
    if avg_abs_corr > 0.7:
        strength = "Very Strong"
    elif avg_abs_corr > 0.5:
        strength = "Strong"
    elif avg_abs_corr > 0.3:
        strength = "Moderate"
    else:
        strength = "Weak"

    correlation_analysis.append({
        'feature': feature,
        'heating_corr': heating_corr,
        'cooling_corr': cooling_corr,
        'avg_abs_corr': avg_abs_corr,
        'strength': strength
    })

    print(f"{feature:<25} {heating_corr:>+8.4f}      {cooling_corr:>+8.4f}      {avg_abs_corr:>6.4f}   {strength:<12}")

# Sort by correlation strength and provide insights
correlation_analysis.sort(key=lambda x: x['avg_abs_corr'], reverse=True)

print(f"\n🏆 RANKING BY CORRELATION STRENGTH:")
for i, item in enumerate(correlation_analysis, 1):
    print(f"{i}. {item['feature']:<25} |r|avg = {item['avg_abs_corr']:.3f} ({item['strength']})")

print(f"\n💡 PHYSICAL INTERPRETATION:")
top_feature = correlation_analysis[0]
print(f"• Strongest predictor: {top_feature['feature']}")

if 'Compactness' in top_feature['feature']:
    print(f"  → More compact buildings minimize surface-to-volume ratio")
    print(f"  → Reduces heat transfer area, improving energy efficiency")
elif 'Surface_Area' in top_feature['feature']:
    print(f"  → Larger surface area increases heat exchange with environment")
    print(f"  → More envelope area = greater energy loss/gain potential")
elif 'Height' in top_feature['feature']:
    print(f"  → Building height affects thermal stratification and air circulation")
    print(f"  → Influences natural ventilation and temperature gradients")

# Enhanced correlation matrix visualization with focus on energy loads
plt.figure(figsize=(12, 10))
feature_energy_corr = energy_df[numeric_features + ['Heating_Load', 'Cooling_Load']].corr()

# Create mask to highlight energy load correlations
mask = np.zeros_like(feature_energy_corr)
mask[:-2, :-2] = True  # Mask out feature-feature correlations to focus on energy relationships

sns.heatmap(feature_energy_corr, annot=True, cmap='RdBu_r', center=0,
           square=True, linewidths=0.5, mask=mask, fmt='.3f',
           cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Feature Correlations with Energy Loads\n(Focus on Heating/Cooling Load Relationships)')
plt.tight_layout()
plt.show()


## Step 39: Unusual Building Configuration Analysis


In [ ]:
"""
STEP 39: UNUSUAL BUILDING CONFIGURATIONS
========================================
Requirement 10: Identify unusual building configurations and extreme cases
"""

print("=" * 80)
print("🔍 REQUIREMENT 10: UNUSUAL BUILDING CONFIGURATIONS")
print("=" * 80)

def identify_outliers_iqr(data, feature, multiplier=1.5):
    """Identify outliers using IQR method"""
    Q1 = data[feature].quantile(0.25)
    Q3 = data[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR

    outliers = data[(data[feature] < lower_bound) | (data[feature] > upper_bound)]
    return outliers, lower_bound, upper_bound

print("🏢 UNUSUAL BUILDING CONFIGURATION ANALYSIS:")
print("─" * 50)

# Use the original numerical features for outlier detection
unusual_configs = {}
for feature in numerical_features:
    outliers, lower, upper = identify_outliers_iqr(energy_df, feature)
    unusual_configs[feature] = {
        'count': len(outliers),
        'percentage': len(outliers) / len(energy_df) * 100,
        'range': (lower, upper),
        'outliers': outliers
    }

    print(f"{feature}:")
    print(f"  Normal range: [{lower:.2f}, {upper:.2f}]")
    print(f"  Unusual configs: {len(outliers)} buildings ({len(outliers)/len(energy_df)*100:.1f}%)")

    if len(outliers) > 0:
        extreme_values = outliers[feature].tolist()
        print(f"  Extreme values: {extreme_values[:5]}{'...' if len(extreme_values) > 5 else ''}")

# Focus on extreme height buildings
height_outliers = unusual_configs['Overall_Height']['outliers']
if len(height_outliers) > 0:
    print(f"\n🏗️ EXTREME HEIGHT BUILDINGS ANALYSIS:")
    print(f"Found {len(height_outliers)} buildings with unusual heights")

    for idx, building in height_outliers.head(3).iterrows():
        print(f"\nBuilding #{idx}:")
        print(f"  Height: {building['Overall_Height']:.1f}")
        print(f"  Compactness: {building['Relative_Compactness']:.3f}")
        print(f"  Surface Area: {building['Surface_Area']:.1f}")
        print(f"  Heating Load: {building['Heating_Load']:.1f}")
        print(f"  Cooling Load: {building['Cooling_Load']:.1f}")

        # Energy efficiency assessment
        avg_heating = energy_df['Heating_Load'].mean()
        avg_cooling = energy_df['Cooling_Load'].mean()
        efficiency = "Efficient" if (building['Heating_Load'] < avg_heating and
                                   building['Cooling_Load'] < avg_cooling) else "Inefficient"
        print(f"  Energy Assessment: {efficiency}")

# Analyze engineered features for unusual configurations (using energy_encoded)
print(f"\n🔬 ENGINEERED FEATURE ANALYSIS:")
engineered_unusual = {}
for eng_feature in ['Total_Area', 'Volume_Proxy', 'Compactness_Height_Ratio']:
    outliers, lower, upper = identify_outliers_iqr(energy_encoded, eng_feature)
    engineered_unusual[eng_feature] = {
        'count': len(outliers),
        'percentage': len(outliers) / len(energy_encoded) * 100,
        'range': (lower, upper)
    }

    print(f"{eng_feature}:")
    print(f"  Normal range: [{lower:.2f}, {upper:.2f}]")
    print(f"  Unusual configs: {len(outliers)} buildings ({len(outliers)/len(energy_encoded)*100:.1f}%)")

# Comprehensive visualization of unusual configurations
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.ravel()

# Plot original features
for i, feature in enumerate(numerical_features):
    if i < len(axes):
        outliers = unusual_configs[feature]['outliers']

        # Box plot with outliers highlighted
        axes[i].boxplot(energy_df[feature], patch_artist=True,
                       boxprops=dict(facecolor='lightblue'))

        if len(outliers) > 0:
            axes[i].scatter([1] * len(outliers), outliers[feature],
                           color='red', s=50, alpha=0.7, label=f'{len(outliers)} outliers')

        axes[i].set_title(f'{feature}\nOutliers: {len(outliers)} ({len(outliers)/len(energy_df)*100:.1f}%)')
        axes[i].set_ylabel('Value')
        if len(outliers) > 0:
            axes[i].legend()

# Plot engineered features
for i, eng_feature in enumerate(['Total_Area', 'Volume_Proxy', 'Compactness_Height_Ratio'], 6):
    if i < len(axes):
        outliers, _, _ = identify_outliers_iqr(energy_encoded, eng_feature)

        axes[i].boxplot(energy_encoded[eng_feature], patch_artist=True,
                       boxprops=dict(facecolor='lightgreen'))

        if len(outliers) > 0:
            axes[i].scatter([1] * len(outliers), outliers[eng_feature],
                           color='red', s=50, alpha=0.7, label=f'{len(outliers)} outliers')

        axes[i].set_title(f'{eng_feature}\nOutliers: {len(outliers)} ({len(outliers)/len(energy_encoded)*100:.1f}%)')
        axes[i].set_ylabel('Value')
        if len(outliers) > 0:
            axes[i].legend()

plt.suptitle('Unusual Building Configurations Analysis\n(Red dots = Outliers)', fontsize=16)
plt.tight_layout()
plt.show()

# Multi-dimensional unusual configuration analysis
print(f"\n🎯 MULTI-DIMENSIONAL UNUSUAL CONFIGURATION ANALYSIS:")

# Identify buildings that are outliers in multiple dimensions
multi_outlier_buildings = set()
for feature, config in unusual_configs.items():
    if len(config['outliers']) > 0:
        multi_outlier_buildings.update(config['outliers'].index.tolist())

print(f"Buildings with unusual configurations in any dimension: {len(multi_outlier_buildings)}")

# Find buildings that are outliers in multiple features
outlier_counts = {}
for building_idx in multi_outlier_buildings:
    count = 0
    for feature, config in unusual_configs.items():
        if building_idx in config['outliers'].index:
            count += 1
    outlier_counts[building_idx] = count

# Buildings with multiple unusual characteristics
extreme_buildings = {idx: count for idx, count in outlier_counts.items() if count >= 2}
print(f"Buildings with multiple unusual characteristics: {len(extreme_buildings)}")

if len(extreme_buildings) > 0:
    print(f"\n🏠 TOP 5 MOST UNUSUAL BUILDINGS:")
    sorted_extreme = sorted(extreme_buildings.items(), key=lambda x: x[1], reverse=True)[:5]

    for idx, outlier_count in sorted_extreme:
        building = energy_df.loc[idx]
        print(f"\nBuilding #{idx} (unusual in {outlier_count} dimensions):")
        print(f"  Height: {building['Overall_Height']:.1f}")
        print(f"  Compactness: {building['Relative_Compactness']:.3f}")
        print(f"  Surface Area: {building['Surface_Area']:.1f}")
        print(f"  Heating Load: {building['Heating_Load']:.1f}")
        print(f"  Cooling Load: {building['Cooling_Load']:.1f}")

# Scatter plot showing relationship between unusual characteristics
plt.figure(figsize=(12, 8))
normal_buildings = energy_df.index.difference(list(multi_outlier_buildings))

plt.scatter(energy_df.loc[normal_buildings, 'Overall_Height'],
           energy_df.loc[normal_buildings, 'Surface_Area'],
           alpha=0.5, color='blue', label='Normal Buildings', s=30)

if len(multi_outlier_buildings) > 0:
    plt.scatter(energy_df.loc[list(multi_outlier_buildings), 'Overall_Height'],
               energy_df.loc[list(multi_outlier_buildings), 'Surface_Area'],
               color='red', label='Unusual Buildings', s=60, alpha=0.8)

plt.xlabel('Overall Height')
plt.ylabel('Surface Area')
plt.title('Unusual Building Configurations: Height vs Surface Area')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\n💡 KEY INSIGHTS ON UNUSUAL CONFIGURATIONS:")
most_outliers_feature = max(unusual_configs.keys(), key=lambda x: unusual_configs[x]['count'])
print(f"• Feature with most outliers: {most_outliers_feature} ({unusual_configs[most_outliers_feature]['count']} buildings)")
print(f"• Total unusual buildings (any dimension): {len(multi_outlier_buildings)} ({len(multi_outlier_buildings)/len(energy_df)*100:.1f}%)")
if len(extreme_buildings) > 0:
    print(f"• Buildings unusual in multiple dimensions: {len(extreme_buildings)} ({len(extreme_buildings)/len(energy_df)*100:.1f}%)")
print(f"• These unusual configurations may represent:")
print(f"  - Specialized building types (e.g., industrial, high-rise)")
print(f"  - Architectural innovations or unique designs")
print(f"  - Potential data collection errors")
print(f"  - Buildings requiring special energy modeling considerations")


# PART 2: WINE QUALITY ASSESSMENT ANALYSIS


## Step 29: Wine Dataset - Load Data


In [ ]:
"""
STEP 29: LOAD WINE DATA
=======================
"""
print("=" * 60)
print("PART 2: WINE QUALITY ASSESSMENT ANALYSIS")
print("=" * 60)

# Load Wine Quality Dataset from UCI repository.
print("Loading Wine Quality Dataset...")
try:
    wine_df = pd.read_csv(
        'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv',
        sep=';'
    )
    print(f"✓ Dataset loaded successfully! Shape: {wine_df.shape}")  # (rows, columns)
except Exception as e:
    print(f"✗ Error loading dataset: {e}")

# .info() prints dtypes, non-null counts; useful for null/memory checks.
print("\nDataset Info:")
print(wine_df.info())

# Quick peek at first 5 rows to confirm parsing and column names.
print("\nFirst 5 rows:")
print(wine_df.head())


## Step 30: Wine Dataset - Data Quality Analysis


In [ ]:
"""
STEP 30: WINE DATA QUALITY ANALYSIS
===================================
"""
# Basic data quality diagnostics for the wine dataset.

print("--- Data Quality Analysis ---")

# .isnull().sum() counts nulls per column; should be all zeros in this dataset.
print(f"Missing values:\n{wine_df.isnull().sum()}")

# .duplicated().sum() counts rows with identical values across all columns.
print(f"\nDuplicate rows: {wine_df.duplicated().sum()}")

# Target variable ('quality') distribution to understand class balance (for regression it's ordinal/continuous-like).
print(f"\n--- Target Variable Analysis ---")
print(f"Quality score distribution:\n{wine_df['quality'].value_counts().sort_index()}")
print(f"Quality range: {wine_df['quality'].min()} to {wine_df['quality'].max()}")
print(f"Quality mean: {wine_df['quality'].mean():.2f} ± {wine_df['quality'].std():.2f}")


## Step 31: Wine Dataset - Quality Distribution Visualization


In [ ]:
"""
STEP 31: WINE QUALITY DISTRIBUTION VISUALIZATION
================================================
"""
# Visualize target ('quality') distribution in two ways for complementary insight.

plt.figure(figsize=(12, 6))         # 12x6 inches overall canvas

# Left subplot: count of each discrete 'quality' label.
plt.subplot(1, 2, 1)
# seaborn.countplot parameters:
#   data=wine_df       → source DataFrame
#   x='quality'        → column whose distinct values to count on the x-axis
sns.countplot(data=wine_df, x='quality')
plt.title('Wine Quality Distribution')
plt.xlabel('Quality Score')
plt.ylabel('Count')

# Right subplot: histogram approximates distribution as continuous; KDE overlays a smooth density.
plt.subplot(1, 2, 2)
# seaborn.histplot parameters:
#   data series → wine_df['quality'] (1D array-like)
#   kde=True    → add kernel density estimate curve
#   bins=6      → number of histogram bins; here aligns roughly to range of integer scores
sns.histplot(wine_df['quality'], kde=True, bins=6)
plt.title('Wine Quality Distribution (Histogram)')
plt.xlabel('Quality Score')
plt.ylabel('Frequency')

plt.tight_layout()  # avoid overlaps between subplots
plt.show()


## Step 32: Wine Dataset - Statistical Summary


In [ ]:
"""
STEP 32: WINE STATISTICAL SUMMARY
=================================
"""
# Print descriptive statistics across all numeric columns to check scale, spread, and potential outliers.

print("--- Statistical Summary ---")
print(wine_df.describe())


## Step 33: Wine Dataset - Feature Distribution Analysis


In [ ]:
"""
STEP 33: WINE FEATURE DISTRIBUTION ANALYSIS
===========================================
"""
# Distribution plots for all input features (excluding target 'quality') in a grid layout.

# Build a list of feature columns (predictors only).
wine_features = [col for col in wine_df.columns if col != 'quality']

n_features = len(wine_features)      # total number of predictors
n_cols = 4                           # number of subplots per row
# ceil division to compute needed rows: (n_features + n_cols - 1) // n_cols
n_rows = (n_features + n_cols - 1) // n_cols

# plt.subplots returns (figure, axes) with specified grid; figsize scales with rows
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.ravel()  # flatten 2D array to 1D for easy indexing

for i, feat in enumerate(wine_features):
    if i < len(axes):
        # seaborn.histplot parameters:
        #   x=wine_df[feat] numeric series
        #   kde=True  → overlay smooth density curve
        #   ax=axes[i]→ draw on i-th subplot
        sns.histplot(wine_df[feat], kde=True, ax=axes[i])
        axes[i].set_title(f'Distribution of {feat}')
        axes[i].tick_params(axis='x', rotation=45)  # rotate tick labels to reduce overlap

# If grid has extra empty axes (when features % n_cols != 0), hide them for a clean look.
for i in range(len(wine_features), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Step 34: Wine Correlation Analysis


In [ ]:
"""
STEP 34: WINE CORRELATION ANALYSIS
==================================
"""
# Feature correlation matrix to identify linear associations and potential multicollinearity.

plt.figure(figsize=(12, 10))

# .corr() computes Pearson correlation by default for numeric columns only.
wine_corr = wine_df.corr()

# seaborn.heatmap parameters:
#   data=wine_corr    → correlation matrix (square DataFrame)
#   annot=True        → write correlation coefficients in each cell
#   cmap='RdBu_r'     → diverging colormap; red/blue with reversed orientation
#   center=0          → white midpoint at zero correlation
#   square=True       → force square cells for aesthetics
#   linewidths=0.5    → thin lines between cells for readability
sns.heatmap(wine_corr, annot=True, cmap='RdBu_r', center=0, square=True, linewidths=0.5)

plt.title('Wine Features Correlation Matrix')
plt.tight_layout()
plt.show()

## Step 35: Wine Quality Correlation Analysis


In [ ]:
"""
STEP 35: WINE QUALITY CORRELATION ANALYSIS
==========================================
"""
# Print correlation of each feature with the target 'quality' to see rough linear influence.

print("--- Quality Correlation Analysis ---")
# wine_df.corr()['quality'] → selects the 'quality' column of the correlation matrix
quality_corr = wine_df.corr()['quality'].sort_values(ascending=False)
print("Features most correlated with quality:")
for feature, corr_val in quality_corr.items():
    if feature != 'quality':
        print(f"  {feature}: {corr_val:.4f}")

## Step 36: Wine Dataset - Outlier Detection


In [ ]:
"""
STEP 36: WINE OUTLIER DETECTION
===============================
"""
# Outlier inspection using boxplots

print("--- Outlier Analysis ---")

fig, axes = plt.subplots(3, 4, figsize=(16, 12))  # 3 rows × 4 cols grid
axes = axes.ravel()

for i, feat in enumerate(wine_features):
    if i < len(axes):
        # seaborn.boxplot parameters:
        #   y=wine_df[feat] → vertical box for feature values
        #   ax=axes[i]      → draw on i-th subplot
        # Behavior (boxplot anatomy) identical to earlier explanation in Energy section.
        sns.boxplot(y=wine_df[feat], ax=axes[i])
        axes[i].set_title(f'Boxplot: {feat}')

plt.tight_layout()
plt.show()


## Step 37: Wine Dataset - Remove Outliers


In [ ]:
"""
STEP 37: WINE REMOVE OUTLIERS
=============================
"""
# Outlier removal helper using the IQR (Interquartile Range) rule.

def remove_outliers_iqr(df, columns):
    """Remove outliers using the 1.5 * IQR rule for specified columns."""
    df_clean = df.copy()     # work on a copy to avoid mutating original frame
    outliers_removed = 0     # counter for removed rows (across all specified columns)

    for column in columns:
        # .quantile(0.25/0.75) gives Q1 and Q3 for the column
        Q1 = df_clean[column].quantile(0.25)
        Q3 = df_clean[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR   # typical Tukey rule lower fence
        upper_bound = Q3 + 1.5 * IQR   # typical Tukey rule upper fence

        before_count = len(df_clean)
        # Keep only rows within [lower_bound, upper_bound] for this column.
        # Note: filtering is cumulative across columns, so rows outlying in any selected
        # column will be dropped.
        df_clean = df_clean[(df_clean[column] >= lower_bound) & (df_clean[column] <= upper_bound)]
        after_count = len(df_clean)
        outliers_removed += before_count - after_count

    return df_clean, outliers_removed

# Choose a subset of columns known to have long tails; reduces extreme leverage in regression.
outlier_features = ['residual sugar', 'free sulfur dioxide', 'total sulfur dioxide']

# Apply function and report the number of removed rows plus new shape.
wine_clean, outliers_count = remove_outliers_iqr(wine_df, outlier_features)
print(f"✓ Outliers removed: {outliers_count}")
print(f"Dataset shape after outlier removal: {wine_clean.shape}")

## Step 38: Wine Dataset - Feature Engineering


In [ ]:
"""
STEP 38: WINE FEATURE ENGINEERING
=================================
Feature engineering adds new predictive signals based on domain knowledge.
For wine quality prediction, we create:
1. Ratios: Indicate balance between components (e.g., free vs. total sulfur dioxide)
2. Composite indicators: Summarize multiple related features (e.g., total acidity)
3. Interaction terms: Capture combined effects of features

WHY FEATURE ENGINEERING?
• Improve model performance by providing more informative features
• Help models capture underlying data patterns and relationships
• Reduce the risk of underfitting by expanding the feature space
"""
print("--- Feature Engineering ---")
wine_engineered = wine_clean.copy()  # non-destructive copy for adding columns

# acid_ratio: fixed / volatile acidity
#  - Higher fixed acidity with lower volatile acidity could indicate different sensory profiles.
wine_engineered['acid_ratio'] = wine_engineered['fixed acidity'] / wine_engineered['volatile acidity']

# sulfur_ratio: free / total sulfur dioxide
#  - Captures proportion of active preservative (free SO2) relative to total; could relate to quality/stability.
wine_engineered['sulfur_ratio'] = wine_engineered['free sulfur dioxide'] / wine_engineered['total sulfur dioxide']

# total_acidity: simple sum of key acidity components
#  - Composite indicator of acidity that might relate to perceived taste and quality.
wine_engineered['total_acidity'] = (
    wine_engineered['fixed acidity'] +
    wine_engineered['volatile acidity'] +
    wine_engineered['citric acid']
)

print("✓ Engineered features created:")
print("- acid_ratio = fixed acidity / volatile acidity")
print("- sulfur_ratio = free sulfur dioxide / total sulfur dioxide")
print("- total_acidity = fixed + volatile + citric acid")

## Step 39: Wine Dataset - Prepare for Modeling


In [ ]:
"""
STEP 39: WINE PREPARE FOR MODELING
==================================
With feature engineering complete, we prepare the wine dataset for modeling:
1. Handle infinite values: Replace ±inf with NaN, then impute NaNs
2. Split data: Separate features (X) and target (y)
3. Train-test split: Split data into training and testing sets
4. Feature scaling: Standardize features to have mean=0, std=1
"""
# Build training data (X) and target (y) for wine; handle infs from division, then split.

# All columns except 'quality' are predictors.
wine_features_for_model = [col for col in wine_engineered.columns if col != 'quality']

X_wine = wine_engineered[wine_features_for_model]  # feature matrix
y_wine = wine_engineered['quality']                # target: quality score

# Division may produce ±inf (e.g., division by zero) and NaNs.
# Replace infinities with NaN, then impute NaNs with column medians.
X_wine = X_wine.replace([np.inf, -np.inf], np.nan)
X_wine = X_wine.fillna(X_wine.median())

print(f"Features for modeling: {len(wine_features_for_model)}")
print(f"Target variable: quality")

# train_test_split parameters:
#   test_size=0.2            → 20% holdout
#   random_state=STUDENT_SEED→ reproducibility
#   stratify=y_wine          → ensures target distribution is similar in train/test (useful here as quality is discrete)
X_wine_train, X_wine_test, y_wine_train, y_wine_test = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=STUDENT_SEED, stratify=y_wine
)

print(f"Wine training set shape: {X_wine_train.shape}")
print(f"Wine test set shape: {X_wine_test.shape}")

# Feature scaling
"""
WHY FEATURE SCALING?
• Standardization (Z-score scaling) centers features to have mean=0 and std=1
• Important for algorithms that rely on distance calculations (e.g., KNN, SVM)
• Helps gradient-based optimizers converge faster
• Regularization methods (e.g., Ridge, Lasso) perform better with standardized data
"""
# Create a new StandardScaler instance.
#   • This will compute mean (μ) and standard deviation (σ) of each feature
#     from the training set only.
#   • Then it transforms values into standardized form: z = (x - μ) / σ
scaler_wine = StandardScaler()

# Fit scaler on training data and transform it in one step.
# .fit_transform does two operations:
#   (1) .fit(X_wine_train) → calculate μ and σ for every column of X_wine_train
#   (2) .transform(X_wine_train) → apply (x - μ) / σ using those stats
X_wine_train_scaled = scaler_wine.fit_transform(X_wine_train)

# Transform the test set using μ and σ from training to avoid leakage:
# .transform ensures the test set uses the same scaling parameters
# as the training set, preventing data leakage.
X_wine_test_scaled = scaler_wine.transform(X_wine_test)

print("✓ Wine features scaled using StandardScaler")

# Sanity check: look at the first few means and standard deviations of scaled training set.

# X_wine_train_scaled.mean(axis=0)
#   • .mean() calculates the average value.
#   • axis=0 → compute mean along rows (per column/feature).
#   • Result: 1D array, one mean for each feature after scaling.
# [:5]
#   • Take only the first 5 values for preview instead of printing all.
# .round(4)
#   • Round each mean to 4 decimal places for neat printing.
print(f"Training set mean (first 5 features): {X_wine_train_scaled.mean(axis=0)[:5].round(4)}")

# X_wine_train_scaled.std(axis=0)
#   • .std() calculates standard deviation.
#   • axis=0 → compute std for each column/feature.
#   • Result: 1D array, one std per feature.
# [:5] → preview only first 5 values.
# .round(4) → round to 4 decimals.
print(f"Training set std (first 5 features): {X_wine_train_scaled.std(axis=0)[:5].round(4)}")

## Step 41: Wine Dataset - Train Linear Regression


In [ ]:
"""
STEP 41: WINE TRAIN LINEAR REGRESSION
=====================================
"""
# Model Training for Wine Dataset

print("--- Wine Model Training and Evaluation ---")

# Container (dict) to store model objects and their evaluation metrics keyed by a label.
wine_results = {}

print("Training Linear Regression for Wine...")
# LinearRegression() uses ordinary least squares (no regularization).
wine_lr = LinearRegression()

# Evaluate on the scaled design matrices for numeric stability.
wine_results['Linear'] = evaluate_model(
    wine_lr,
    X_wine_train_scaled,   # features: scaled train
    X_wine_test_scaled,    # features: scaled test
    y_wine_train,          # targets: train (kept unscaled for interpretability)
    y_wine_test,           # targets: test
    'Linear Regression'    # label for tracking/reporting
)

# f-strings with format specifiers:
#   {value:.4f} → fixed 4 decimal places
print(f"✓ Wine Linear Regression - R²: {wine_results['Linear']['r2']:.4f}, RMSE: {wine_results['Linear']['rmse']:.4f}")


## Step 42: Wine Dataset - Train Polynomial Regression


In [ ]:
"""
STEP 42: WINE TRAIN POLYNOMIAL REGRESSION
=========================================
"""
# Polynomial Regression for Wine using a Pipeline:
# Pipeline structure: [('poly', PolynomialFeatures), ('scaler', StandardScaler), ('reg', LinearRegression)]
for degree in [2, 3, 4]:
    print(f"Training Polynomial Regression (degree {degree}) for Wine...")

    # PolynomialFeatures parameters:
    #   degree=degree      : include all polynomial terms up to the specified degree (incl. interactions/powers)
    #   include_bias=False : do NOT add constant 1 column (LinearRegression has its own intercept)
    # StandardScaler()     : standardize expanded feature space
    # LinearRegression()   : OLS on transformed features
    wine_poly_pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ])

    # IMPORTANT:
    # Pass the RAW X (unscaled) because the pipeline includes its own scaler after polynomial expansion.
    wine_results[f'Poly_{degree}'] = evaluate_model(
        wine_poly_pipeline,
        X_wine_train, X_wine_test,
        y_wine_train, y_wine_test,
        f'Polynomial {degree}'
    )

    print(f"✓ Wine Polynomial {degree} - R²: {wine_results[f'Poly_{degree}']['r2']:.4f}, "
          f"RMSE: {wine_results[f'Poly_{degree}']['rmse']:.4f}")


## Step 43: Wine Dataset - Results Summary


In [ ]:
"""
STEP 43: WINE RESULTS SUMMARY
=============================
"""
# Summarize Wine model performance and determine the best by R² (higher is better).

print("--- Wine Quality Results ---")
# Nice fixed-width header: column labels with left alignment and width specs
print(f"{'Model':<20} {'R²':<8} {'RMSE':<8} {'MAE':<8} {'MSE':<10}")
print("-" * 60)

# Iterate dict items; format floats to 4 decimals for readability
for model_name, results in wine_results.items():
    print(f"{model_name:<20} {results['r2']:<8.4f} {results['rmse']:<8.4f} "
          f"{results['mae']:<8.4f} {results['mse']:<10.4f}")

# Find best wine model
wine_best = max(wine_results.keys(), key=lambda x: wine_results[x]['r2'])
print(f"\n🏆 Best Wine Model: {wine_best} with R² = {wine_results[wine_best]['r2']:.4f}")


## Step 44: Wine Dataset - Feature Importance and Visualization


In [ ]:
"""
STEP 44: WINE FEATURE IMPORTANCE AND VISUALIZATION
=================================================
"""
# Wine Feature importance analysis

print("--- Feature Importance Analysis ---")

# Build a DataFrame mapping each input feature (wine_features_for_model) to the corresponding
# linear coefficient learned by the baseline Linear Regression on scaled features.
wine_feature_importance = pd.DataFrame({
    'feature': wine_features_for_model,
    'coefficient': wine_results['Linear']['model'].coef_
})

# Add absolute value column to rank by magnitude irrespective of sign.
wine_feature_importance['abs_coefficient'] = np.abs(wine_feature_importance['coefficient'])

# Sort descending so the most influential features are at the top.
wine_feature_importance = wine_feature_importance.sort_values('abs_coefficient', ascending=False)

plt.figure(figsize=(12, 8))  # overall canvas size for two subplots

# ── Subplot 1: Top-10 absolute coefficients bar plot ─────────────────────────
plt.subplot(1, 2, 1)  # (nrows=1, ncols=2, index=1 → left plot)
top_wine_features = wine_feature_importance.head(10)

# seaborn.barplot parameters:
#   data=top_wine_features  : DataFrame providing columns to map
#   x='abs_coefficient'     : numeric values (bar lengths)
#   y='feature'             : category labels (y-axis)
sns.barplot(data=top_wine_features, x='abs_coefficient', y='feature')

plt.title('Top 10 Wine Feature Importance')
plt.xlabel('Absolute Coefficient Value')                    # axis label (units: scaled)

# ── Subplot 2: Predicted vs Actual scatter for the best wine model ──────────
plt.subplot(1, 2, 2)  # index=2 → right plot

# Retrieve cached predictions from the best model entry in wine_results.
best_wine_predictions = wine_results[wine_best]['predictions']

# plt.scatter parameters:
#   x=y_wine_test                → true target values on x-axis
#   y=best_wine_predictions      → predicted target values on y-axis
#   alpha=0.6                    → semi-transparency so dense areas show density
plt.scatter(y_wine_test, best_wine_predictions, alpha=0.6)

# Identity (perfect prediction) reference line:
#   [min, max] for both x and y to span the observed range; 'r--' is red dashed line style; lw=2 is line width 2 points
plt.plot([y_wine_test.min(), y_wine_test.max()],
         [y_wine_test.min(), y_wine_test.max()],
         'r--', lw=2)

plt.xlabel('Actual Wine Quality')
plt.ylabel('Predicted Wine Quality')
plt.title(f'Wine: Predicted vs Actual ({wine_best})')

plt.tight_layout()  # adjust spacing to avoid overlap
plt.show()


# PART 3: MUSIC RELEASE YEAR PREDICTION ANALYSIS


## Step 45: Music Dataset - Load Data


In [ ]:
"""
STEP 45: LOAD MUSIC DATA WITH INTELLIGENT CACHING
=================================================
This step implements a comprehensive data loading strategy with multiple fallback options.
The music dataset is very large (515k samples, 90 features), so we use caching to avoid
repeated downloads and processing.

LOADING STRATEGY HIERARCHY:
1. Try processed cache (fastest - pickled DataFrame)
2. Try local CSV file (medium - raw text parsing)
3. Try cached ZIP extraction (medium - decompress then parse)
4. Download from UCI repository (slowest - network dependent)
5. Create synthetic data (fallback - for demonstration)
"""
print("="*60)
print("PART 3: MUSIC RELEASE YEAR PREDICTION ANALYSIS")
print("="*60)

print("Loading Music Release Year Dataset...")
print("Note: This is a large dataset (515k samples), using intelligent caching...")

# Define file paths for multi-level caching system
# Each file represents a different stage of data preparation
music_zip_file = 'YearPredictionMSD.txt.zip'     # Original compressed download
music_csv_file = 'YearPredictionMSD.txt'         # Extracted raw CSV data
music_processed_file = 'music_dataset_processed.pkl'  # Processed DataFrame with column names

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Try to load processed data first (fastest option)
# ─────────────────────────────────────────────────────────────────────────────
"""
WHY START WITH PROCESSED DATA?
• Pickled DataFrames load 10-100x faster than CSV parsing
• Column names and data types are already set correctly
• No need for additional preprocessing steps
• Best user experience for repeat runs
"""
try:
    if os.path.exists(music_processed_file):
        print(f"✓ Loading from processed cache: {music_processed_file}")
        # pd.read_pickle() deserializes the entire DataFrame object
        # This preserves column names, data types, and index information
        music_df = pd.read_pickle(music_processed_file)
        print(f"✓ Cached dataset loaded successfully! Shape: {music_df.shape}")
        load_success = True
    else:
        load_success = False
except Exception as e:
    print(f"⚠️ Could not load cached data: {e}")
    load_success = False

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: If no cache, try to load from local CSV file
# ─────────────────────────────────────────────────────────────────────────────
"""
WHY TRY LOCAL CSV SECOND?
• Avoids network dependency if file was previously extracted
• Faster than downloading but slower than pickle
• Requires column name assignment since original has no header
"""
if not load_success:
    try:
        if os.path.exists(music_csv_file):
            print(f"✓ Loading from local CSV: {music_csv_file}")
            # header=None because the original dataset has no column headers
            # We'll assign meaningful names later in Step 6
            music_df = pd.read_csv(music_csv_file, header=None)
            print(f"✓ Local CSV loaded successfully! Shape: {music_df.shape}")
            load_success = True
        else:
            load_success = False
    except Exception as e:
        print(f"⚠️ Could not load local CSV: {e}")
        load_success = False

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: If no local CSV, try to extract from cached ZIP file
# ─────────────────────────────────────────────────────────────────────────────
"""
WHY TRY ZIP EXTRACTION THIRD?
• ZIP file might exist from previous download attempt
• Avoids re-downloading large file over network
• Extracts to current directory for future use
"""
if not load_success:
    try:
        if os.path.exists(music_zip_file):
            print(f"✓ Extracting from cached ZIP: {music_zip_file}")
            import zipfile

            # Extract all files from ZIP to current directory
            # with statement ensures proper file handle cleanup
            with zipfile.ZipFile(music_zip_file, 'r') as zip_ref:
                zip_ref.extractall('.')  # '.' means current directory

            # Now try to load the freshly extracted CSV file
            music_df = pd.read_csv(music_csv_file, header=None)
            print(f"✓ ZIP extracted and loaded successfully! Shape: {music_df.shape}")
            load_success = True
        else:
            load_success = False
    except Exception as e:
        print(f"⚠️ Could not extract from ZIP: {e}")
        load_success = False

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: If no local files, download from UCI repository
# ─────────────────────────────────────────────────────────────────────────────
"""
WHY DOWNLOAD AS LAST RESORT?
• Network-dependent and potentially slow
• Large file (~50MB compressed)
• May fail due to connectivity or server issues
• But necessary for first-time users
"""
if not load_success:
    try:
        print("📥 Downloading from UCI repository (this may take several minutes)...")
        music_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00203/YearPredictionMSD.txt.zip'

        # Stream download to handle large files efficiently
        # stream=True prevents loading entire file into memory at once
        response = requests.get(music_url, stream=True)
        response.raise_for_status()  # Raises HTTPError for bad responses (4xx or 5xx)

        # Get total file size from headers for progress tracking
        total_size = int(response.headers.get('content-length', 0))

        # Download file in chunks to provide progress feedback
        with open(music_zip_file, 'wb') as f:
            downloaded_size = 0
            # iter_content yields chunks of specified size
            # chunk_size=8192 (8KB) is efficient for most network conditions
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                downloaded_size += len(chunk)

                # Simple progress indicator (overwrites same line)
                if total_size > 0:
                    progress = downloaded_size / total_size * 100
                    print(f"\rDownload progress: {progress:.1f}%", end="", flush=True)

        print(f"\n✓ ZIP file downloaded: {music_zip_file}")

        # Extract the downloaded ZIP file immediately
        import zipfile
        with zipfile.ZipFile(music_zip_file, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"✓ ZIP file extracted: {music_csv_file}")

        # Load the extracted data
        music_df = pd.read_csv(music_csv_file, header=None)
        print(f"✓ Dataset downloaded and loaded successfully! Shape: {music_df.shape}")
        load_success = True

    except Exception as e:
        print(f"✗ Error downloading dataset: {e}")
        load_success = False

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Fallback to synthetic data if all else fails
# ─────────────────────────────────────────────────────────────────────────────
"""
WHY SYNTHETIC DATA FALLBACK?
• Ensures code can run even without internet/data access
• Useful for testing and demonstration purposes
• Maintains same data structure as real dataset
• Educational - shows data shape and column structure
"""
if not load_success:
    print("⚠️ All loading methods failed. Creating synthetic data for demonstration...")

    # Set seed for reproducible synthetic data
    np.random.seed(STUDENT_SEED)
    n_samples = 10000  # Smaller than real dataset for speed

    # Create DataFrame with same structure as real dataset
    # 91 columns total: 1 year + 90 audio features
    music_df = pd.DataFrame(np.random.randn(n_samples, 91))

    # First column is year (realistic range 1960-2011)
    music_df.iloc[:, 0] = np.random.randint(1960, 2012, n_samples)

    print(f"✓ Synthetic dataset created! Shape: {music_df.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: Set column names and save processed data for future use
# ─────────────────────────────────────────────────────────────────────────────
"""
COLUMN NAMING STRATEGY:
• First column: 'year' (target variable - what we want to predict)
• Remaining 90 columns: 'feature_1' through 'feature_90' (audio characteristics)

REAL DATASET FEATURE DESCRIPTION:
• Features 1-12: Timbre average values (spectral characteristics)
• Features 13-90: Timbre covariance values (spectral relationships)
• All features are extracted from audio analysis of songs
"""
# Set consistent, meaningful column names
music_df.columns = ['year'] + [f'feature_{i}' for i in range(1, 91)]

# Save processed data for future runs (if we loaded real data successfully)
# This creates the fastest-loading cache for subsequent runs
if load_success and not os.path.exists(music_processed_file):
    try:
        # to_pickle() serializes the entire DataFrame efficiently
        # Preserves data types, column names, and index
        music_df.to_pickle(music_processed_file)
        print(f"✓ Processed data cached for future use: {music_processed_file}")
    except Exception as e:
        print(f"⚠️ Could not cache processed data: {e}")

# ─────────────────────────────────────────────────────────────────────────────
# FINAL VERIFICATION AND REPORTING
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n📊 DATASET SUMMARY:")
print(f"   Shape: {music_df.shape}")  # (samples, features)
print(f"   Year range: {music_df['year'].min()} to {music_df['year'].max()}")
print(f"   Features: {len([col for col in music_df.columns if col.startswith('feature_')])} audio features")
# Calculate memory usage in MB for performance awareness
print(f"   Size: {music_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB in memory")

# Display file status for user awareness and troubleshooting
print(f"\n📁 LOCAL FILE STATUS:")
files_to_check = [music_zip_file, music_csv_file, music_processed_file]
for file_path in files_to_check:
    if os.path.exists(file_path):
        # Get file size in MB for storage awareness
        file_size = os.path.getsize(file_path) / 1024**2
        print(f"   ✓ {file_path}: {file_size:.1f} MB")
    else:
        print(f"   ✗ {file_path}: Not found")

# User guidance for optimization and troubleshooting
print(f"\n💡 OPTIMIZATION NOTES:")
print(f"   • Next run will load from cache (much faster)")
print(f"   • To force re-download, delete: {music_zip_file}")
print(f"   • To clear all cache, delete: {', '.join(files_to_check)}")


## Step 46: Music Dataset - Sampling for Efficiency


In [ ]:
"""
STEP 46: MUSIC SAMPLING FOR COMPUTATIONAL EFFICIENCY
====================================================
The full music dataset has 515k samples, which creates computational challenges:
• Long training times (hours instead of minutes)
• High memory usage (gigabytes of RAM)
• Difficult to experiment and iterate quickly

SAMPLING STRATEGY:
• Use systematic sampling to maintain data distribution
• Choose sample size based on available computational resources
• Ensure sample is large enough for statistical validity
"""
print("--- Sampling for Computational Efficiency ---")

# Determine optimal sample size based on dataset size and resources
# min() ensures we don't try to sample more data than available
sample_size = min(5000, len(music_df))  # Cap at 50k samples for analysis

# Count non-null values in any column (usually same as total rows if no missing data)
total_samples = music_df['year'].count()
print(f"Total samples: {total_samples}")

"""
WHY 50,000 SAMPLES?
• Large enough for statistical significance (central limit theorem)
• Small enough for reasonable computation time (minutes vs hours)
• Maintains good representation of temporal distribution
• Allows for multiple experimental iterations

RANDOM SAMPLING BENEFITS:
• random_state=STUDENT_SEED ensures reproducibility
• Each student gets consistent results across runs
• Maintains approximately same distribution as full dataset
"""
music_sample = music_df.sample(n=sample_size, random_state=STUDENT_SEED)
print(f"✓ Using sample of {len(music_sample)} records for analysis")

# Quick verification that sampling preserved data distribution
print(f"   Original year range: {music_df['year'].min()}-{music_df['year'].max()}")
print(f"   Sample year range: {music_sample['year'].min()}-{music_sample['year'].max()}")
print(f"   Sampling ratio: {len(music_sample)/len(music_df)*100:.1f}% of original data")


## Step 47: Music Dataset - Temporal Analysis


In [ ]:
"""
STEP 47: MUSIC TEMPORAL ANALYSIS
================================
Understanding the temporal distribution is crucial because:
• We're predicting YEAR, so time is our target variable
• Music technology and styles evolved significantly over decades
• Data collection biases may favor certain eras
• Model performance may vary across different time periods

ANALYSIS COMPONENTS:
1. Overall year distribution (histogram)
2. Decade-based grouping (bar chart)
3. Year variability within decades (box plot)
"""
print("--- Temporal Analysis ---")

# Create comprehensive temporal visualization
plt.figure(figsize=(15, 6))  # Wide figure to accommodate three subplots

# ── SUBPLOT 1: Year Distribution ──────────────────────────────────────────
plt.subplot(1, 3, 1)
"""
HISTOGRAM ANALYSIS:
• bins=30 creates 30 equal-width intervals across the year range
• kde=True adds a smooth density curve showing the underlying distribution
• This reveals whether data is uniformly distributed across years
"""
sns.histplot(music_sample['year'], bins=30, kde=True)
plt.title('Year Distribution')
plt.xlabel('Release Year')
plt.ylabel('Frequency')

# ── SUBPLOT 2: Decade Aggregation ──────────────────────────────────────────
plt.subplot(1, 3, 2)
"""
DECADE ANALYSIS LOGIC:
• (year // 10) * 10 converts years to decade start years
• Example: 1987 → (1987 // 10) * 10 → 1980
• This groups 1980-1989 as "1980s", 1990-1999 as "1990s", etc.
"""
# Create decade feature by truncating to decade start year
music_sample['decade'] = (music_sample['year'] // 10) * 10

# Count songs per decade and sort chronologically
decade_counts = music_sample['decade'].value_counts().sort_index()

# Create bar chart showing song distribution by decade
plt.bar(decade_counts.index, decade_counts.values)
plt.title('Songs by Decade')
plt.xlabel('Decade')
plt.ylabel('Number of Songs')

# ── SUBPLOT 3: Decade Variability ──────────────────────────────────────────
plt.subplot(1, 3, 3)
"""
BOX PLOT ANALYSIS:
• Shows year distribution WITHIN each decade
• Box = interquartile range (25th to 75th percentile)
• Line in box = median
• Whiskers = extend to min/max within 1.5*IQR
• Dots = outliers beyond whiskers

WHY THIS MATTERS:
• Reveals if data is evenly distributed within decades
• Identifies decades with uneven temporal sampling
• Shows potential clustering or gaps in data collection
"""
# Create list of year arrays, one per decade for box plotting
decade_years = [
    music_sample[music_sample['decade'] == decade]['year']
    for decade in sorted(decade_counts.index)
]

plt.boxplot(decade_years)
plt.title('Year Distribution by Decade')
plt.xlabel('Decade')
plt.ylabel('Release Year')
# Set x-axis labels to decade names (e.g., "1980s")
plt.xticks(range(1, len(decade_counts) + 1),
           [f"{int(d)}s" for d in sorted(decade_counts.index)])

plt.tight_layout()  # Prevent overlapping labels
plt.show()

# ── NUMERICAL SUMMARY OF TEMPORAL PATTERNS ──────────────────────────────────
print(f"\n📊 TEMPORAL DISTRIBUTION INSIGHTS:")
print(f"   • Total decades covered: {len(decade_counts)}")
print(f"   • Most represented decade: {int(decade_counts.idxmax())}s ({decade_counts.max()} songs)")
print(f"   • Least represented decade: {int(decade_counts.idxmin())}s ({decade_counts.min()} songs)")

# Calculate coefficient of variation to measure distribution evenness
cv = decade_counts.std() / decade_counts.mean()
print(f"   • Distribution evenness (CV): {cv:.2f} ({'uneven' if cv > 0.5 else 'moderate' if cv > 0.2 else 'even'})")

print(f"\n💡 IMPLICATIONS FOR MODELING:")
if cv > 0.5:
    print(f"   ⚠️  High temporal bias detected - model may favor well-represented decades")
    print(f"   💡 Consider: stratified sampling or weighted loss functions")
else:
    print(f"   ✓ Reasonable temporal distribution for unbiased modeling")


## Step 48: Music Dataset - Data Quality Analysis


In [ ]:
"""
STEP 48: MUSIC DATA QUALITY ANALYSIS
====================================
Data quality is critical for reliable machine learning results.
Poor quality data leads to:
• Unreliable model predictions
• Misleading performance metrics
• Invalid scientific conclusions

QUALITY CHECKS PERFORMED:
1. Missing values (NaN, null entries)
2. Duplicate rows (exact copies)
3. Invalid years (outside reasonable range)
4. Feature value ranges and distributions
"""
print("--- Music Data Quality Analysis ---")

# ── CHECK 1: Missing Values ───────────────────────────────────────────────
"""
MISSING VALUE ANALYSIS:
• .isnull() returns boolean DataFrame (True where data is missing)
• .sum().sum() counts total missing values across all columns
• Missing data can indicate collection errors or preprocessing issues
"""
total_missing = music_sample.isnull().sum().sum()
print(f"Missing values: {total_missing} total missing values")

if total_missing > 0:
    print(f"   Missing values by column:")
    missing_by_column = music_sample.isnull().sum()
    for col, missing_count in missing_by_column[missing_by_column > 0].items():
        print(f"     {col}: {missing_count} missing ({missing_count/len(music_sample)*100:.1f}%)")

# ── CHECK 2: Duplicate Rows ───────────────────────────────────────────────
"""
DUPLICATE DETECTION:
• .duplicated() returns boolean Series (True for duplicate rows)
• Considers ALL columns when determining duplicates
• Duplicates can indicate data collection errors or multiple sources
"""
duplicate_count = music_sample.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

if duplicate_count > 0:
    print(f"   📊 {duplicate_count} duplicate rows found ({duplicate_count/len(music_sample)*100:.1f}% of data)")
    print(f"   💡 Consider: Remove duplicates or investigate data collection process")

# ── CHECK 3: Invalid Years ────────────────────────────────────────────────
"""
YEAR VALIDATION LOGIC:
• Music recording technology: ~1920 onwards (early phonograph era)
• Dataset cutoff: ~2015 (when this dataset was created)
• Years outside this range likely indicate data errors

WHY THESE BOUNDS?
• < 1920: Pre-commercial recording era, very rare
• > 2015: Future dates impossible at dataset creation time
"""
invalid_years = ((music_sample['year'] < 1920) | (music_sample['year'] > 2015)).sum()
print(f"Invalid years (outside 1920-2015): {invalid_years}")

if invalid_years > 0:
    # Show specific invalid years for debugging
    invalid_year_values = music_sample[
        (music_sample['year'] < 1920) | (music_sample['year'] > 2015)
    ]['year'].unique()
    print(f"   Invalid year values found: {sorted(invalid_year_values)}")

# ── CHECK 4: Feature Value Ranges ─────────────────────────────────────────
"""
AUDIO FEATURE VALIDATION:
• Audio features should have reasonable ranges
• Extreme outliers may indicate processing errors
• Consistent scales across features improve model stability
"""
print(f"\n📊 FEATURE VALUE RANGE ANALYSIS:")

# Get all audio feature columns
feature_cols = [col for col in music_sample.columns if col.startswith('feature_')]

# Calculate basic statistics for feature validation
feature_stats = music_sample[feature_cols].describe()

# Check for extreme values that might indicate errors
print(f"   Feature range summary (first 5 features):")
for i, col in enumerate(feature_cols[:5]):
    min_val = feature_stats.loc['min', col]
    max_val = feature_stats.loc['max', col]
    range_val = max_val - min_val
    print(f"     {col}: [{min_val:.3f}, {max_val:.3f}] (range: {range_val:.3f})")

# Check for features with suspicious characteristics
suspicious_features = []
for col in feature_cols:
    # Check for zero variance (constant features)
    if feature_stats.loc['std', col] < 1e-10:
        suspicious_features.append(f"{col}: near-zero variance")

    # Check for extreme ranges that might indicate scaling issues
    range_val = feature_stats.loc['max', col] - feature_stats.loc['min', col]
    if range_val > 1000:  # Arbitrary threshold for "very large range"
        suspicious_features.append(f"{col}: very large range ({range_val:.1f})")

if suspicious_features:
    print(f"\n⚠️  POTENTIALLY SUSPICIOUS FEATURES:")
    for issue in suspicious_features[:5]:  # Show first 5 issues
        print(f"   • {issue}")
    if len(suspicious_features) > 5:
        print(f"   • ... and {len(suspicious_features) - 5} more issues")

# ── OVERALL DATA QUALITY ASSESSMENT ──────────────────────────────────────
print(f"\n🎯 DATA QUALITY SUMMARY:")

quality_score = 100  # Start with perfect score
issues = []

if total_missing > 0:
    missing_percent = total_missing / (len(music_sample) * len(music_sample.columns)) * 100
    quality_score -= missing_percent * 2  # Penalty for missing data
    issues.append(f"Missing data: {missing_percent:.1f}%")

if duplicate_count > 0:
    dup_percent = duplicate_count / len(music_sample) * 100
    quality_score -= dup_percent
    issues.append(f"Duplicates: {dup_percent:.1f}%")

if invalid_years > 0:
    invalid_percent = invalid_years / len(music_sample) * 100
    quality_score -= invalid_percent * 3  # Higher penalty for invalid targets
    issues.append(f"Invalid years: {invalid_percent:.1f}%")

quality_score = max(0, quality_score)  # Don't go below 0

print(f"   📊 Overall quality score: {quality_score:.1f}/100")

if quality_score > 90:
    print(f"   ✅ Excellent data quality - ready for modeling")
elif quality_score > 70:
    print(f"   ⚠️  Good data quality with minor issues")
    for issue in issues:
        print(f"     • {issue}")
else:
    print(f"   ❌ Data quality concerns detected")
    for issue in issues:
        print(f"     • {issue}")
    print(f"   💡 Recommend: Data cleaning before modeling")


## Step 49: Music Dataset - Feature Analysis


In [ ]:
"""
STEP 49: MUSIC FEATURE ANALYSIS
===============================
Understanding audio features is crucial for:
• Interpreting model results and predictions
• Identifying the most important characteristics for year prediction
• Detecting potential data quality issues in feature extraction
• Understanding what makes music from different eras distinctive

AUDIO FEATURE STRUCTURE:
The 90 audio features follow a specific pattern based on timbre analysis:
• Features 1-12: Timbre average values (spectral centroids)
• Features 13-90: Timbre covariance values (spectral relationships)
"""
print("--- Feature Analysis ---")

# Get all audio feature column names for analysis
feature_cols = [col for col in music_sample.columns if col.startswith('feature_')]
print(f"Number of audio features: {len(feature_cols)}")

# ── TIMBRE FEATURE CATEGORIZATION ─────────────────────────────────────────
"""
TIMBRE ANALYSIS BACKGROUND:
Timbre describes the 'color' or 'texture' of sound - what makes a piano
sound different from a guitar playing the same note.

FEATURE BREAKDOWN:
• Timbre averages (1-12): Mean spectral characteristics over time
  - Represent overall 'brightness', 'warmth', etc. of the audio
  - Analogous to average color in an image

• Timbre covariances (13-90): Relationships between spectral components
  - Capture how different frequency bands interact
  - Analogous to texture patterns in an image
"""

# Separate features into meaningful categories for analysis
timbre_avg_features = feature_cols[:12]    # Features 1-12: averages
timbre_cov_features = feature_cols[12:]    # Features 13-90: covariances

print(f"\nFeature categorization:")
print(f"   • Timbre averages (features 1-12): {len(timbre_avg_features)} features")
print(f"   • Timbre covariances (features 13-90): {len(timbre_cov_features)} features")

# ── STATISTICAL ANALYSIS OF TIMBRE AVERAGES ──────────────────────────────
print(f"\n📊 TIMBRE AVERAGE STATISTICS (Features 1-12):")
"""
WHY ANALYZE TIMBRE AVERAGES FIRST?
• More interpretable than covariance features
• Directly relate to perceptual audio qualities
• Often the most predictive for music classification tasks
• Easier to spot data quality issues
"""
timbre_avg_stats = music_sample[timbre_avg_features].describe()
print(timbre_avg_stats)

# Analyze the range and variability of timbre averages
print(f"\n📈 TIMBRE AVERAGE INSIGHTS:")
for i, feature in enumerate(timbre_avg_features):
    mean_val = timbre_avg_stats.loc['mean', feature]
    std_val = timbre_avg_stats.loc['std', feature]
    range_val = timbre_avg_stats.loc['max', feature] - timbre_avg_stats.loc['min', feature]

    # Calculate coefficient of variation to assess relative variability
    cv = std_val / abs(mean_val) if abs(mean_val) > 1e-10 else float('inf')

    # Provide interpretation based on statistical properties
    if cv < 0.2:
        variability = "Low variability - stable characteristic"
    elif cv < 0.5:
        variability = "Moderate variability - somewhat diverse"
    else:
        variability = "High variability - very diverse characteristic"

    print(f"   {feature}: Mean={mean_val:6.3f}, Std={std_val:5.3f}, Range={range_val:6.3f} ({variability})")

# ── STATISTICAL ANALYSIS OF TIMBRE COVARIANCES ───────────────────────────
print(f"\n📊 TIMBRE COVARIANCE STATISTICS (Features 13-90):")
"""
COVARIANCE FEATURE CHARACTERISTICS:
• Typically smaller values than averages (often near zero)
• Higher variability due to interaction effects
• More sensitive to audio processing artifacts
• Can be correlated with each other
"""
timbre_cov_stats = music_sample[timbre_cov_features].describe()
print(timbre_cov_stats)

# Compare average magnitudes between feature types
avg_timbre_avg_magnitude = abs(music_sample[timbre_avg_features]).mean().mean()
avg_timbre_cov_magnitude = abs(music_sample[timbre_cov_features]).mean().mean()

print(f"\n🔍 FEATURE TYPE COMPARISON:")
print(f"   Average |magnitude| of timbre averages: {avg_timbre_avg_magnitude:.6f}")
print(f"   Average |magnitude| of timbre covariances: {avg_timbre_cov_magnitude:.6f}")
print(f"   Magnitude ratio (avg/cov): {avg_timbre_avg_magnitude/avg_timbre_cov_magnitude:.2f}")

# ── FEATURE CORRELATION ANALYSIS ─────────────────────────────────────────
"""
WHY CHECK FEATURE CORRELATIONS?
• High correlation = redundant information
• Can cause numerical instability in linear models
• Helps identify which features to prioritize
• Reveals underlying structure in audio characteristics
"""
print(f"\n🔗 FEATURE INTERCORRELATION ANALYSIS:")

# Calculate correlation matrix for a subset of features (computationally expensive for all 90)
sample_features = timbre_avg_features + timbre_cov_features[:10]  # 22 features total
correlation_subset = music_sample[sample_features].corr()

# Find highly correlated feature pairs
high_corr_pairs = []
for i in range(len(sample_features)):
    for j in range(i+1, len(sample_features)):
        corr_val = correlation_subset.iloc[i, j]
        if abs(corr_val) > 0.7:  # Threshold for "high correlation"
            high_corr_pairs.append((sample_features[i], sample_features[j], corr_val))

if high_corr_pairs:
    print(f"   High correlation pairs found:")
    for feat1, feat2, corr in high_corr_pairs[:5]:  # Show first 5
        print(f"     {feat1} ↔ {feat2}: r = {corr:.3f}")
    if len(high_corr_pairs) > 5:
        print(f"     ... and {len(high_corr_pairs) - 5} more pairs")
else:
    print(f"   ✓ No highly correlated feature pairs detected (|r| < 0.7)")

# ── FEATURE QUALITY ASSESSMENT ───────────────────────────────────────────
print(f"\n🎯 AUDIO FEATURE QUALITY ASSESSMENT:")

# Check for potential issues in feature extraction
issues_found = []

# 1. Check for features with very low variance (potentially uninformative)
low_variance_features = []
for feature in feature_cols:
    if music_sample[feature].std() < 0.001:
        low_variance_features.append(feature)

if low_variance_features:
    issues_found.append(f"Low variance features: {len(low_variance_features)}")

# 2. Check for features with extreme outliers
outlier_features = []
for feature in feature_cols:
    # Using IQR method to detect extreme outliers
    Q1 = music_sample[feature].quantile(0.25)
    Q3 = music_sample[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 3 * IQR  # 3*IQR = extreme outliers
    upper_bound = Q3 + 3 * IQR

    outliers = ((music_sample[feature] < lower_bound) | (music_sample[feature] > upper_bound)).sum()
    if outliers > len(music_sample) * 0.05:  # > 5% outliers
        outlier_features.append(feature)

if outlier_features:
    issues_found.append(f"Features with many outliers: {len(outlier_features)}")

# 3. Overall assessment
if not issues_found:
    print(f"   ✅ Audio features appear well-behaved and suitable for modeling")
else:
    print(f"   ⚠️ Potential feature quality issues detected:")
    for issue in issues_found:
        print(f"     • {issue}")

print(f"\n💡 FEATURE INSIGHTS FOR MODELING:")
print(f"   • {len(feature_cols)} audio features provide rich representation of musical characteristics")
print(f"   • Timbre averages likely more interpretable and stable than covariances")
print(f"   • Feature selection may help focus on most predictive characteristics")
print(f"   • Scaling will be important due to different feature magnitudes")


## Step 50: Music Dataset - Feature Distribution Visualization


In [ ]:
"""
STEP 50: MUSIC FEATURE DISTRIBUTION VISUALIZATION
=================================================
Visual analysis of feature distributions reveals:
• Whether features follow normal distributions (important for linear models)
• Presence of outliers or unusual patterns
• Skewness that might need transformation
• Relative scales between different features

FOCUS ON TIMBRE AVERAGES:
We visualize only the first 12 features (timbre averages) because:
• More interpretable than covariance features
• Manageable number for visual inspection
• Represent core spectral characteristics
• Most likely to show clear patterns
"""
print("--- Feature Distribution Visualization ---")

# Create comprehensive visualization grid for timbre average features
fig, axes = plt.subplots(3, 4, figsize=(16, 12))  # 3 rows × 4 columns = 12 plots
axes = axes.ravel()  # Flatten 2D array to 1D for easy indexing

# ── DISTRIBUTION PLOTS FOR EACH TIMBRE AVERAGE FEATURE ──────────────────
"""
HISTOGRAM + KDE COMBINATION:
• Histogram bars: Show actual frequency distribution (discrete)
• KDE curve: Show smooth probability density estimate (continuous)
• Together: Reveal both discrete patterns and underlying continuous distribution

WHAT TO LOOK FOR:
• Bell curves = normal distribution (good for linear models)
• Skewed distributions = might need transformation
• Multiple peaks = potential subgroups in the data
• Heavy tails = outliers or extreme values
"""
for i, feat in enumerate(timbre_avg_features):
    # seaborn.histplot parameters:
    #   kde=True → overlay kernel density estimate
    #   ax=axes[i] → specify which subplot to draw on
    #   color and alpha → consistent styling
    sns.histplot(music_sample[feat], kde=True, ax=axes[i],
                color='skyblue', alpha=0.7, edgecolor='black')

    # Customize each subplot
    plt.title(f'Distribution of {feat}', fontsize=12, fontweight='bold')
    plt.xlabel(f'{feat} Value')
    plt.ylabel('Frequency')
    axes[i].grid(True, alpha=0.3)  # Light grid for easier reading

    # Add statistical annotations for quick reference
    mean_val = music_sample[feat].mean()
    std_val = music_sample[feat].std()

    # Add vertical line at mean for reference
    axes[i].axvline(mean_val, color='red', linestyle='--', alpha=0.7, linewidth=2)

    # Add text annotation with basic stats
    axes[i].text(0.02, 0.98, f'μ={mean_val:.3f}\nσ={std_val:.3f}',
                transform=axes[i].transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                fontsize=9)

# Add overall title for the entire figure
plt.suptitle('Timbre Average Features Distribution Analysis\n(Red dashed line = Mean)',
            fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout()  # Prevent overlapping labels
plt.show()

# ── STATISTICAL SUMMARY OF DISTRIBUTION CHARACTERISTICS ────────────────
print(f"\n📊 DISTRIBUTION ANALYSIS SUMMARY:")

distribution_summary = []

for feature in timbre_avg_features:
    data = music_sample[feature]

    # Calculate distribution characteristics
    mean_val = data.mean()
    median_val = data.median()
    std_val = data.std()
    skewness = data.skew()  # Measure of asymmetry
    kurtosis = data.kurtosis()  # Measure of tail heaviness

    # Classify distribution shape based on skewness
    if abs(skewness) < 0.5:
        shape = "Approximately Normal"
    elif skewness > 0.5:
        shape = "Right-skewed (long right tail)"
    else:
        shape = "Left-skewed (long left tail)"

    # Classify tail behavior based on kurtosis
    if kurtosis > 1:
        tails = "Heavy tails (more outliers)"
    elif kurtosis < -1:
        tails = "Light tails (fewer outliers)"
    else:
        tails = "Normal tails"

    distribution_summary.append({
        'feature': feature,
        'mean': mean_val,
        'median': median_val,
        'std': std_val,
        'skewness': skewness,
        'kurtosis': kurtosis,
        'shape': shape,
        'tails': tails
    })

# Display summary table
print(f"{'Feature':<12} {'Mean':<8} {'Median':<8} {'Std':<7} {'Skew':<6} {'Shape':<25}")
print("─" * 80)

for summary in distribution_summary:
    print(f"{summary['feature']:<12} {summary['mean']:<8.3f} {summary['median']:<8.3f} "
          f"{summary['std']:<7.3f} {summary['skewness']:<6.2f} {summary['shape']:<25}")

# ── MODELING IMPLICATIONS ────────────────────────────────────────────────
print(f"\n💡 IMPLICATIONS FOR MODELING:")

# Count features by distribution type
normal_count = sum(1 for s in distribution_summary if "Normal" in s['shape'])
skewed_count = len(distribution_summary) - normal_count

print(f"   📈 Distribution types:")
print(f"     • Approximately normal: {normal_count}/{len(distribution_summary)} features")
print(f"     • Skewed distributions: {skewed_count}/{len(distribution_summary)} features")

if normal_count > skewed_count:
    print(f"   ✅ Most features are approximately normal - good for linear models")
else:
    print(f"   ⚠️ Many features are skewed - consider transformations")
    print(f"     💡 Potential fixes: log transform, Box-Cox, StandardScaler")

# Check for extreme skewness that might need attention
very_skewed = [s for s in distribution_summary if abs(s['skewness']) > 1.5]
if very_skewed:
    print(f"   ⚠️ Highly skewed features that may need transformation:")
    for s in very_skewed:
        print(f"     • {s['feature']}: skewness = {s['skewness']:.2f}")

# Check for heavy-tailed distributions
heavy_tailed = [s for s in distribution_summary if s['kurtosis'] > 2]
if heavy_tailed:
    print(f"   ⚠️ Heavy-tailed features (many outliers):")
    for s in heavy_tailed:
        print(f"     • {s['feature']}: kurtosis = {s['kurtosis']:.2f}")

print(f"\n🎯 PREPROCESSING RECOMMENDATIONS:")
print(f"   1. Apply StandardScaler to normalize feature scales")
if skewed_count > 0:
    print(f"   2. Consider robust scaling (RobustScaler) for skewed features")
if very_skewed:
    print(f"   3. Apply log or power transformations to highly skewed features")
if heavy_tailed:
    print(f"   4. Consider outlier removal or robust regression methods")
print(f"   5. Feature selection may help focus on most predictive characteristics")


## Step 51: Music Dataset - Feature Selection


In [ ]:
"""
STEP 51: MUSIC FEATURE SELECTION
================================
With 90 audio features, we face the "curse of dimensionality":
• Too many features relative to samples can cause overfitting
• Irrelevant features add noise and reduce model performance
• Computational complexity increases with feature count
• Model interpretability decreases with too many features

FEATURE SELECTION STRATEGY:
We use SelectKBest with f_regression which:
• Calculates F-statistic for each feature vs target (year)
• Ranks features by their univariate predictive power
• Selects top K features based on statistical significance
• Provides interpretable scoring for feature importance

WHY K=30 FEATURES?
• Balances model complexity with predictive power
• Manageable for interpretation and visualization
• Reduces risk of overfitting on limited sample
• Common rule: use √n or n/10 features where n = samples
"""
print("--- Feature Selection and Engineering ---")

# Prepare features (X) and target (y) for feature selection
X_music = music_sample[feature_cols]  # All 90 audio features
y_music = music_sample['year']        # Target: release year

print(f"Original feature matrix shape: {X_music.shape}")
print(f"Target vector shape: {y_music.shape}")

# ── APPLY UNIVARIATE FEATURE SELECTION ───────────────────────────────────
print(f"\nApplying feature selection...")

k_best_features = 30  # Select top 30 features out of 90

"""
SelectKBest EXPLANATION:
• score_func=f_regression: Uses F-test for regression (continuous target)
  - Alternative: mutual_info_regression for non-linear relationships
  - F-test assumes linear relationship between feature and target
• k=k_best_features: Number of top features to select
• fit_transform(): Fits selector and transforms data in one step
"""
selector = SelectKBest(score_func=f_regression, k=k_best_features)

# Apply feature selection
X_music_selected = selector.fit_transform(X_music, y_music)

print(f"Selected feature matrix shape: {X_music_selected.shape}")
print(f"Dimensionality reduction: {len(feature_cols)} → {X_music_selected.shape[1]} features")

# ── EXTRACT FEATURE SELECTION RESULTS ────────────────────────────────────
"""
GETTING SELECTION RESULTS:
• get_support(indices=True): Returns indices of selected features
• scores_: F-statistic scores for all original features
• Higher F-score = stronger linear relationship with target
• Selects top K features based on statistical significance
"""
# Get indices of selected features
selected_feature_indices = selector.get_support(indices=True)

# Map indices back to feature names for interpretability
selected_features = [feature_cols[i] for i in selected_feature_indices]

# Get F-scores for the selected features
selected_scores = selector.scores_[selected_feature_indices]

print(f"✓ Selected {k_best_features} best features based on F-regression scores")

# ── ANALYZE FEATURE SELECTION PATTERNS ───────────────────────────────────
print(f"\n📊 FEATURE SELECTION ANALYSIS:")

# Count how many timbre averages vs covariances were selected
timbre_avg_selected = [f for f in selected_features if int(f.split('_')[1]) <= 12]
timbre_cov_selected = [f for f in selected_features if int(f.split('_')[1]) > 12]

print(f"   Selected feature breakdown:")
print(f"     • Timbre averages (1-12): {len(timbre_avg_selected)}/{12} features")
print(f"     • Timbre covariances (13-90): {len(timbre_cov_selected)}/{78} features")

# Calculate selection rates for each feature type
avg_selection_rate = len(timbre_avg_selected) / 12 * 100
cov_selection_rate = len(timbre_cov_selected) / 78 * 100

print(f"   Selection rates:")
print(f"     • Timbre averages: {avg_selection_rate:.1f}% selected")
print(f"     • Timbre covariances: {cov_selection_rate:.1f}% selected")

# Interpret the selection pattern
if avg_selection_rate > cov_selection_rate:
    print(f"   💡 Timbre averages are more predictive for year prediction")
    print(f"      → Overall spectral characteristics matter more than interactions")
else:
    print(f"   💡 Timbre covariances are equally/more important")
    print(f"      → Feature interactions crucial for temporal discrimination")

# ── FEATURE SCORE ANALYSIS ───────────────────────────────────────────────
print(f"\n📈 F-SCORE STATISTICS:")

all_scores = selector.scores_
selected_scores = selector.scores_[selected_feature_indices]

print(f"   All features F-scores:")
print(f"     • Range: {all_scores.min():.2f} - {all_scores.max():.2f}")
print(f"     • Mean: {all_scores.mean():.2f}")
print(f"     • Standard deviation: {all_scores.std():.2f}")

print(f"   Selected features F-scores:")
print(f"     • Range: {selected_scores.min():.2f} - {selected_scores.max():.2f}")
print(f"     • Mean: {selected_scores.mean():.2f}")
print(f"     • Minimum selected: {selected_scores.min():.2f}")

# Calculate selection threshold
selection_threshold = selected_scores.min()
print(f"   Selection threshold: F-score ≥ {selection_threshold:.2f}")

# Count how many features were above/below threshold
above_threshold = (all_scores >= selection_threshold).sum()
print(f"   Features above threshold: {above_threshold}/{len(all_scores)}")

# ── QUALITY ASSURANCE CHECKS ─────────────────────────────────────────────
print(f"\n🔍 FEATURE SELECTION QUALITY CHECKS:")

# Check 1: Ensure we got exactly k features
assert X_music_selected.shape[1] == k_best_features, "Feature count mismatch!"
print(f"   ✓ Correct number of features selected: {X_music_selected.shape[1]}")

# Check 2: Ensure no duplicate features
assert len(selected_features) == len(set(selected_features)), "Duplicate features detected!"
print(f"   ✓ No duplicate features in selection")

# Check 3: Check that selected features make sense
feature_numbers = [int(f.split('_')[1]) for f in selected_features]
print(f"   ✓ Selected feature range: {min(feature_numbers)} - {max(feature_numbers)}")

# Check 4: Verify scores are reasonable
if selected_scores.min() > 1.0:  # F-scores should be > 1 for meaningful features
    print(f"   ✓ All selected features have meaningful F-scores (> 1.0)")
else:
    print(f"   ⚠️ Some selected features have low F-scores (< 1.0)")

print(f"\n💡 FEATURE SELECTION INSIGHTS:")
print(f"   • Reduced dimensionality by {(1 - k_best_features/len(feature_cols))*100:.1f}%")
print(f"   • Selected features show strong univariate relationships with year")
print(f"   • {'Timbre averages' if avg_selection_rate > cov_selection_rate else 'Both feature types'} are important for temporal prediction")
print(f"   • Next step: Train models on selected features for better performance")


## Step 53: Music Dataset - Train-Test Split and Scaling


In [ ]:
"""
STEP 53: MUSIC TRAIN-TEST SPLIT AND SCALING
===========================================
With feature selection complete, we prepare the data for modeling:
1. Import required libraries for modeling
2. Define evaluation function
3. Split data: Separate features (X) and target (y) using selected features
4. Train-test split: Split data into training and testing sets
5. Feature scaling: Standardize features to have mean=0, std=1
"""

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, RobustScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.feature_selection import SelectKBest, f_regression
from scipy import stats
from scipy.stats import pearsonr, spearmanr

print("✓ All required libraries imported successfully")

# Define the evaluate_model function
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Comprehensive model evaluation function

    Parameters:
    - model: scikit-learn model instance
    - X_train, X_test: training and test features
    - y_train, y_test: training and test targets
    - model_name: string name for the model

    Returns:
    - Dictionary with model object and all evaluation metrics
    """
    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)

    return {
        'model': model,
        'predictions': y_pred,
        'r2': r2,
        'rmse': rmse,
        'mae': mae,
        'mse': mse
    }

print("✓ evaluate_model function defined successfully")

# Train-test split for Music
X_music_train, X_music_test, y_music_train, y_music_test = train_test_split(
    X_music_selected,  # Use the selected features from Step 51
    y_music,
    test_size=0.2,
    random_state=STUDENT_SEED,
    stratify=None  # No stratification needed for continuous target
)

print(f"Music training set shape: {X_music_train.shape}")
print(f"Music test set shape: {X_music_test.shape}")

# Feature scaling
"""
WHY FEATURE SCALING?
• Standardization (Z-score scaling) centers features to have mean=0 and std=1
• Important for algorithms that rely on distance calculations (e.g., KNN, SVM)
• Helps gradient-based optimizers converge faster
• Regularization methods (e.g., Ridge, Lasso) perform better with standardized data
"""
# Create a new StandardScaler instance.
music_scaler = StandardScaler()

# Fit scaler on training data and transform it in one step.
X_music_train_scaled = music_scaler.fit_transform(X_music_train)

# Transform the test set using μ and σ from training to avoid leakage
X_music_test_scaled = music_scaler.transform(X_music_test)

print("✓ Music features scaled using StandardScaler")

# Verify scaling worked correctly
print(f"Training set mean (first 5 features): {X_music_train_scaled.mean(axis=0)[:5].round(4)}")
print(f"Training set std (first 5 features): {X_music_train_scaled.std(axis=0)[:5].round(4)}")

## Step 54: Music Dataset - Train Linear Regression


In [ ]:
"""
STEP 54: MUSIC TRAIN LINEAR REGRESSION
======================================
Train baseline linear regression model on the music dataset to predict release year
from audio features. This establishes our performance baseline for comparison with
more complex polynomial models.
"""
print("--- Music Model Training and Evaluation ---")

# Initialize results dictionary to store all model performance metrics
music_results = {}

print("Training Linear Regression for Music...")

# Create and train linear regression model
music_lr = LinearRegression()

# Train and evaluate the model using our evaluation function
music_results['Linear'] = evaluate_model(
    music_lr,
    X_music_train_scaled,
    X_music_test_scaled,
    y_music_train,
    y_music_test,
    'Linear Regression'
)

# Display results
print(f"✓ Music Linear Regression - R²: {music_results['Linear']['r2']:.4f}, RMSE: {music_results['Linear']['rmse']:.4f}")

# Additional insight about the baseline performance
baseline_r2 = music_results['Linear']['r2']
baseline_rmse = music_results['Linear']['rmse']

print(f"\n📊 BASELINE PERFORMANCE ANALYSIS:")
print(f"   • R² = {baseline_r2:.4f} means model explains {baseline_r2*100:.2f}% of year variance")
print(f"   • RMSE = {baseline_rmse:.2f} years average prediction error")
print(f"   • For a dataset spanning ~{y_music_test.max() - y_music_test.min():.0f} years, this represents {'good' if baseline_rmse < 10 else 'moderate' if baseline_rmse < 20 else 'poor'} accuracy")

## Step 55: Music Dataset - Enhanced Polynomial Degree Analysis


In [ ]:
"""
STEP 55: ENHANCED POLYNOMIAL DEGREE ANALYSIS
============================================
REQUIREMENT ALIGNMENT: Addresses requirement 2 - Find recommended degree and explain why
"""

# Enhanced polynomial regression testing with more degrees and detailed analysis
print("--- Enhanced Polynomial Degree Analysis ---")

# Test polynomial degrees 1-5 to find optimal degree
polynomial_degrees_extended = [1, 2, 3, 4, 5]
degree_performance = {}

for degree in polynomial_degrees_extended:
    print(f"Training Polynomial Regression (degree {degree}) for Music...")
    try:
        # For high-dimensional data, use interaction_only=True to avoid feature explosion
        # This creates only interaction terms, not pure polynomial powers
        music_poly_pipeline = Pipeline([
            ('poly', PolynomialFeatures(degree=degree, include_bias=False, interaction_only=True)),
            ('scaler', StandardScaler()),
            ('reg', LinearRegression())
        ])

        # Use a smaller subset for computational efficiency with higher degrees
        if degree > 2:
            # Use subset of 10k samples for degrees > 2 to manage computational complexity
            subset_size = 10000
            test_subset_size = 1000 # Use a smaller test set as well
            train_subset_indices = np.random.choice(len(X_music_train), subset_size, replace=False)
            X_subset = X_music_train[train_subset_indices]
            y_subset = y_music_train.iloc[train_subset_indices]

            test_subset_indices = np.random.choice(len(X_music_test), test_subset_size, replace=False)
            X_test_subset = X_music_test[test_subset_indices]
            y_test_subset = y_music_test.iloc[test_subset_indices]

            result = evaluate_model(music_poly_pipeline, X_subset, X_test_subset,
                                  y_subset, y_test_subset, f'Polynomial {degree}')
        else:
            result = evaluate_model(music_poly_pipeline, X_music_train, X_music_test,
                                  y_music_train, y_music_test, f'Polynomial {degree}')

        degree_performance[degree] = result
        print(f"✓ Music Polynomial {degree} - R²: {result['r2']:.4f}, "
              f"RMSE: {result['rmse']:.4f}")

    except Exception as e:
        print(f"✗ Error training Polynomial degree {degree}: {e}")
        degree_performance[degree] = {'r2': -np.inf, 'rmse': np.inf}

# Find optimal degree and explain recommendation
best_degree = max(degree_performance.keys(),
                 key=lambda x: degree_performance[x]['r2'] if degree_performance[x]['r2'] != -np.inf else -np.inf)

print(f"\n🎯 RECOMMENDED DEGREE: {best_degree}")
print(f"   R² = {degree_performance[best_degree]['r2']:.4f}")
print(f"   RMSE = {degree_performance[best_degree]['rmse']:.4f}")

print(f"\n💡 WHY DEGREE {best_degree} IS RECOMMENDED:")
if best_degree == 1:
    print("• Linear relationships dominate in audio-year prediction")
    print("• Higher degrees lead to overfitting with 90-dimensional feature space")
    print("• Temporal trends in music are primarily linear over decades")
elif best_degree == 2:
    print("• Quadratic terms capture subtle non-linear audio evolution")
    print("• Balances complexity with generalization for temporal prediction")
    print("• Interaction terms reveal audio feature combinations important for era detection")
else:
    print(f"• Complex audio patterns require degree {best_degree} interactions")
    print("• Musical evolution has intricate non-linear relationships")
    print("• Feature interactions become crucial for year discrimination")

print(f"\n⚠️ HIGH-DIMENSIONAL CHALLENGES:")
print("• 90 features × polynomial degree creates feature explosion")
print("• Degree 3+ with 90 features → thousands of polynomial terms")
print("• Risk of overfitting increases exponentially with degree")
print("• Computational complexity grows as O(n^degree)")


## Step 56: Music Dataset - Temporal Feature Evolution Analysis


In [ ]:
"""
STEP 56: MUSIC TEMPORAL FEATURE EVOLUTION ANALYSIS
==================================================
"""
print("--- Temporal Feature Evolution Analysis ---")

# Analyze how audio features evolved over decades
print("Analyzing how audio features changed over time...")

# Group data by decades for temporal analysis
music_sample['decade'] = (music_sample['year'] // 10) * 10
decade_groups = music_sample.groupby('decade')

# Calculate standard deviations for each decade (this was missing)
decade_stds = []
decades = sorted(music_sample['decade'].unique())

for decade in decades:
    decade_data = music_sample[music_sample['decade'] == decade][selected_features]
    decade_std = decade_data.std().mean()  # Average std across all features for this decade
    decade_stds.append(decade_std)

# Calculate correlations between features and year
feature_names = selected_features
year_correlations = []
for feature in feature_names:
    corr, _ = pearsonr(music_sample[feature], music_sample['year'])
    year_correlations.append(abs(corr))  # Use absolute correlation

year_correlations = np.array(year_correlations)

# Create visualization of temporal evolution
plt.figure(figsize=(15, 10))

# Subplot 1: Feature correlations with year
plt.subplot(2, 2, 1)
top_10_indices = np.argsort(year_correlations)[-10:]
top_10_features = [feature_names[i] for i in top_10_indices]
top_10_correlations = year_correlations[top_10_indices]

plt.barh(range(len(top_10_features)), top_10_correlations)
plt.yticks(range(len(top_10_features)), top_10_features)
plt.xlabel('Absolute Correlation with Year')
plt.title('Top 10 Features Most Correlated with Year')

# Subplot 2: Decade variability evolution
plt.subplot(2, 2, 2)
plt.plot(decades, decade_stds, marker='o', linewidth=2, markersize=6)
plt.xlabel('Decade')
plt.ylabel('Average Feature Standard Deviation')
plt.title('Feature Variability Evolution Over Time')
plt.grid(True, alpha=0.3)

# Additional analysis
plt.subplot(2, 2, 3)
# Show average feature values by decade for top predictive feature
best_feature = feature_names[np.argmax(year_correlations)]
decade_means = []
for decade in decades:
    decade_mean = music_sample[music_sample['decade'] == decade][best_feature].mean()
    decade_means.append(decade_mean)

plt.plot(decades, decade_means, marker='s', linewidth=2, markersize=6, color='red')
plt.xlabel('Decade')
plt.ylabel(f'Average {best_feature}')
plt.title(f'Evolution of Most Predictive Feature: {best_feature}')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print insights
print(f"\n🎵 TEMPORAL EVOLUTION INSIGHTS:")
print(f"• FEATURE PREDICTABILITY: {len([c for c in year_correlations if c > 0.1])} features show strong temporal correlation")
print(f"• MODERN COMPLEXITY: Audio features show {'increasing' if year_correlations[0] > 0 else 'decreasing'} complexity over time")
print(f"• STRONGEST TEMPORAL PREDICTOR: {feature_names[np.argmax(year_correlations)]} (|r| = {max(year_correlations):.3f})")
print(f"• FEATURE VARIANCE: {'Higher' if decade_stds[-1] > decade_stds[0] else 'Lower'} variability in recent decades")
print(f"• TECHNOLOGICAL IMPACT: Digital music production affects timbre characteristics")

print(f"\n💡 IMPLICATIONS FOR YEAR PREDICTION:")
if max(year_correlations) > 0.3:
    print("• Strong temporal patterns suggest good predictability")
    print("• Linear models should perform reasonably well")
else:
    print("• Weak temporal patterns suggest challenging prediction task")
    print("• May need more complex models or feature engineering")

if decade_stds[-1] > decade_stds[0]:
    print("• Increasing variability over time may challenge model generalization")
else:
    print("• Stable variability over time should help model generalization")

## Step 57: Music Dataset - Feature Importance and Temporal Analysis


In [ ]:
"""
STEP 57: MUSIC FEATURE IMPORTANCE AND TEMPORAL ANALYSIS
=======================================================
"""
print("--- Music Feature Importance and Temporal Analysis ---")

# Create feature importance DataFrame from linear regression coefficients
if 'Linear' in music_results:
    # Get the trained linear regression model
    linear_model = music_results['Linear']['model']

    # Create feature importance DataFrame
    music_feature_importance = pd.DataFrame({
        'feature': selected_features,
        'coefficient': linear_model.coef_
    })

    # Add absolute coefficient column for ranking
    music_feature_importance['abs_coefficient'] = np.abs(music_feature_importance['coefficient'])

    # Sort by absolute importance
    music_feature_importance = music_feature_importance.sort_values('abs_coefficient', ascending=False)

    print("✓ Feature importance analysis created from linear regression coefficients")
else:
    print("❌ Linear regression results not found - cannot analyze feature importance")
    music_feature_importance = pd.DataFrame()

# Continue with the rest of the analysis
if not music_feature_importance.empty:
    linear_importance = music_feature_importance.copy()

    print(f"\n🏆 TOP 10 LINEAR PREDICTORS:")
    print(f"{'Rank':<5} {'Feature':<15} {'Coefficient':<12} {'Abs. Coeff':<12} {'Type':<20}")
    print("─" * 75)

    for i, (idx, row) in enumerate(linear_importance.head(10).iterrows()):
        feature_num = int(row['feature'].split('_')[1])
        feature_type = "Timbre Average" if feature_num <= 12 else "Timbre Covariance"

        print(f"{i+1:<5} {row['feature']:<15} {row['coefficient']:<12.6f} {row['abs_coefficient']:<12.6f} {feature_type:<20}")

    # Analyze feature importance patterns
    top_features = linear_importance.head(10)
    timbre_avg_count = sum(1 for f in top_features['feature'] if int(f.split('_')[1]) <= 12)
    timbre_cov_count = 10 - timbre_avg_count

    print(f"\n📊 IMPORTANCE PATTERN ANALYSIS:")
    print(f"   • Timbre averages in top 10: {timbre_avg_count}/10")
    print(f"   • Timbre covariances in top 10: {timbre_cov_count}/10")

    if timbre_avg_count > timbre_cov_count:
        print("   → Timbre averages dominate temporal prediction")
    else:
        print("   → Mixed importance between feature types")

    # Create visualization
    plt.figure(figsize=(15, 10))

    # Subplot 1: Feature importance bar plot
    plt.subplot(2, 2, 1)
    top_10 = linear_importance.head(10)

    colors = ['red' if int(f.split('_')[1]) <= 12 else 'blue' for f in top_10['feature']]
    bars = plt.barh(range(len(top_10)), top_10['abs_coefficient'], color=colors, alpha=0.7)

    plt.yticks(range(len(top_10)), top_10['feature'])
    plt.xlabel('Absolute Coefficient Value')
    plt.title('Top 10 Most Important Features')
    plt.gca().invert_yaxis()

    # Add legend
    import matplotlib.patches as mpatches
    red_patch = mpatches.Patch(color='red', alpha=0.7, label='Timbre Average')
    blue_patch = mpatches.Patch(color='blue', alpha=0.7, label='Timbre Covariance')
    plt.legend(handles=[red_patch, blue_patch])

    # Continue with temporal correlation analysis and other visualizations...
    plt.tight_layout()
    plt.show()

    print(f"\n💡 FEATURE IMPORTANCE INSIGHTS:")
    print(f"   • Most important feature: {linear_importance.iloc[0]['feature']} (coeff: {linear_importance.iloc[0]['coefficient']:.6f})")
    print(f"   • Feature importance spans {linear_importance['abs_coefficient'].max()/linear_importance['abs_coefficient'].min():.1f}x range")
    print(f"   • {'Spectral averages' if timbre_avg_count > 5 else 'Mixed features'} are most predictive of release year")

else:
    print("❌ Cannot perform feature importance analysis without linear regression results")

## Step 58: Music Dataset - Temporal Bias and Data Distribution Analysis


In [ ]:
"""
STEP 58: TEMPORAL BIAS AND DATA DISTRIBUTION ANALYSIS
=====================================================
REQUIREMENT ALIGNMENT: Addresses requirement 8 - Are there temporal clusters or biases?
"""

print("=" * 80)
print("🎯 REQUIREMENT 8: TEMPORAL CLUSTERS AND BIASES")
print("=" * 80)

# Analyze temporal distribution and potential biases
print("📊 TEMPORAL DATA DISTRIBUTION ANALYSIS:")

# Year distribution statistics
year_counts = music_sample['year'].value_counts().sort_index()
print(f"Data span: {music_sample['year'].min()} - {music_sample['year'].max()}")
print(f"Total years covered: {music_sample['year'].nunique()}")
print(f"Average songs per year: {len(music_sample) / music_sample['year'].nunique():.1f}")

# Decade analysis
decade_distribution = music_sample['decade'].value_counts().sort_index()
print(f"\n🎵 DECADE DISTRIBUTION:")
print(f"{'Decade':<10} {'Count':<8} {'Percentage':<12} {'Bias Level':<15}")
print("─" * 50)

total_songs = len(music_sample)
expected_per_decade = total_songs / len(decade_distribution)

for decade, count in decade_distribution.items():
    percentage = count / total_songs * 100
    bias_ratio = count / expected_per_decade

    if bias_ratio > 1.5:
        bias_level = "High Over-rep"
    elif bias_ratio > 1.2:
        bias_level = "Moderate Over-rep"
    elif bias_ratio < 0.5:
        bias_level = "High Under-rep"
    elif bias_ratio < 0.8:
        bias_level = "Moderate Under-rep"
    else:
        bias_level = "Balanced"

    print(f"{int(decade)}s      {count:<8} {percentage:<12.1f} {bias_level:<15}")

# Identify temporal clusters using year gaps
year_diff = music_sample['year'].value_counts().sort_index()
gaps = []
previous_year = None
for year in year_diff.index:
    if previous_year is not None and year - previous_year > 1:
        gaps.append((previous_year + 1, year - 1))
    previous_year = year

print(f"\n📈 TEMPORAL GAPS DETECTED:")
if gaps:
    for start_gap, end_gap in gaps:
        print(f"• Missing data: {start_gap}-{end_gap} ({end_gap - start_gap + 1} years)")
else:
    print(f"• No significant temporal gaps detected")

# Comprehensive temporal visualization
plt.figure(figsize=(16, 12))

# Plot 1: Year distribution histogram
plt.subplot(3, 2, 1)
plt.hist(music_sample['year'], bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('Release Year')
plt.ylabel('Number of Songs')
plt.title('Song Distribution by Year')
plt.grid(True, alpha=0.3)

# Plot 2: Decade bias visualization
plt.subplot(3, 2, 2)
bias_ratios = [decade_distribution[d] / expected_per_decade for d in sorted(decade_distribution.keys())]
decade_labels = [f"{int(d)}s" for d in sorted(decade_distribution.keys())]
colors = ['red' if b > 1.2 else 'blue' if b < 0.8 else 'green' for b in bias_ratios]
plt.bar(decade_labels, bias_ratios, color=colors, alpha=0.7)
plt.axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
plt.xlabel('Decade')
plt.ylabel('Bias Ratio (Actual/Expected)')
plt.title('Temporal Bias by Decade')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

# Plot 3: Cumulative song count over time
plt.subplot(3, 2, 3)
cumulative_counts = music_sample.groupby('year').size().cumsum()
plt.plot(cumulative_counts.index, cumulative_counts.values, linewidth=2)
plt.xlabel('Year')
plt.ylabel('Cumulative Song Count')
plt.title('Cumulative Music Data Over Time')
plt.grid(True, alpha=0.3)

# Plot 4: Model performance by decade
plt.subplot(3, 2, 4)
if 'Linear' in music_results:
    decade_rmse = {}
    for decade in sorted(decade_distribution.keys()):
        decade_mask = music_sample['decade'] == decade
        if decade_mask.sum() > 10:  # Ensure sufficient data
            decade_data = music_sample[decade_mask]
            # Simple approximation using overall model
            decade_rmse[decade] = np.std(decade_data['year'])  # Use std as proxy for prediction difficulty

    plt.bar([f"{int(d)}s" for d in decade_rmse.keys()], list(decade_rmse.values()), alpha=0.7)
    plt.xlabel('Decade')
    plt.ylabel('Year Variability (Std)')
    plt.title('Prediction Difficulty by Decade')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)

# Plot 5: Feature correlation by era
plt.subplot(3, 2, 5)
early_era = music_sample[music_sample['year'] < 1990]
modern_era = music_sample[music_sample['year'] >= 1990]

if len(early_era) > 100 and len(modern_era) > 100:
    early_corr = [early_era[f].corr(early_era['year']) for f in selected_features[:10]]
    modern_corr = [modern_era[f].corr(modern_era['year']) for f in selected_features[:10]]

    x_pos = range(len(selected_features[:10]))
    width = 0.35
    plt.bar([x - width/2 for x in x_pos], [abs(c) for c in early_corr], width, label='Pre-1990', alpha=0.7)
    plt.bar([x + width/2 for x in x_pos], [abs(c) for c in modern_corr], width, label='Post-1990', alpha=0.7)
    plt.xlabel('Top Features')
    plt.ylabel('|Correlation with Year|')
    plt.title('Feature-Year Correlation by Era')
    plt.xticks(x_pos, [f'F{i+1}' for i in range(10)])
    plt.legend()
    plt.grid(True, alpha=0.3)

# Plot 6: Data quality by year
plt.subplot(3, 2, 6)
yearly_stats = music_sample.groupby('year')[selected_features[0]].agg(['count', 'std']).reset_index()
plt.scatter(yearly_stats['year'], yearly_stats['count'], alpha=0.6, s=yearly_stats['std']*10)
plt.xlabel('Year')
plt.ylabel('Songs per Year')
plt.title('Data Density by Year (Size = Feature Std)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n🎯 TEMPORAL BIAS CONCLUSIONS:")
most_biased_decade = max(decade_distribution.keys(), key=lambda x: decade_distribution[x])
least_biased_decade = min(decade_distribution.keys(), key=lambda x: decade_distribution[x])

print(f"• HIGHEST REPRESENTATION: {int(most_biased_decade)}s with {decade_distribution[most_biased_decade]} songs")
print(f"• LOWEST REPRESENTATION: {int(least_biased_decade)}s with {decade_distribution[least_biased_decade]} songs")
print(f"• BIAS IMPACT: Model likely performs better on {int(most_biased_decade)}s music")
print(f"• RECOMMENDATION: Use stratified sampling or weighted loss functions")


## Step 59: Music Dataset - Feature Addition Impact Analysis


In [ ]:
"""
STEP 59: FEATURE ADDITION IMPACT ANALYSIS
=========================================
REQUIREMENT ALIGNMENT: Addresses requirement 9 - How adding more features impacts metrics
"""

print("=" * 80)
print("🎯 REQUIREMENT 9: IMPACT OF ADDING MORE FEATURES")
print("=" * 80)

# Progressive feature addition analysis
feature_counts = [5, 10, 15, 20, 25, 30]
feature_impact_results = {}

print("📊 PROGRESSIVE FEATURE ADDITION ANALYSIS:")
print(f"{'Features':<10} {'R²':<10} {'RMSE':<10} {'Training Time':<15} {'Change in R²':<12.5}")
print("─" * 75)

for k in feature_counts:
    if k <= len(selected_features):
        print(f"Testing with {k} features...")

        # Select top k features
        selector_k = SelectKBest(score_func=f_regression, k=k)
        X_music_k = selector_k.fit_transform(X_music, y_music)

        # Split and scale
        X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(
            X_music_k, y_music, test_size=0.2, random_state=STUDENT_SEED)

        scaler_k = StandardScaler()
        X_train_k_scaled = scaler_k.fit_transform(X_train_k)
        X_test_k_scaled = scaler_k.transform(X_test_k)

        # Train model and measure time
        import time
        start_time = time.time()

        lr_k = LinearRegression()
        result_k = evaluate_model(lr_k, X_train_k_scaled, X_test_k_scaled,
                                y_train_k, y_test_k, f'Linear_{k}')

        training_time = time.time() - start_time
        feature_impact_results[k] = {
            'r2': result_k['r2'],
            'rmse': result_k['rmse'],
            'time': training_time
        }

        # Calculate change in R²
        if k == feature_counts[0]:
            r2_change = 0
        else:
            prev_k = [x for x in feature_counts if x < k][-1]
            r2_change = result_k['r2'] - feature_impact_results[prev_k]['r2']

        print(f"{k:<10} {result_k['r2']:<10.4f} {result_k['rmse']:<10.4f} {training_time:<15.3f}s {r2_change:<+12.5f}")

# Analysis of diminishing returns
print(f"\n📈 FEATURE ADDITION INSIGHTS:")

# Find optimal feature count (best R² improvement per feature)
if len(feature_impact_results) > 1:
    feature_efficiency = {}
    for i, k in enumerate(feature_counts[1:], 1):
        if k in feature_impact_results and feature_counts[i-1] in feature_impact_results:
            r2_gain = feature_impact_results[k]['r2'] - feature_impact_results[feature_counts[i-1]]['r2']
            features_added = k - feature_counts[i-1]
            efficiency = r2_gain / features_added
            feature_efficiency[k] = efficiency

    if feature_efficiency:
        most_efficient = max(feature_efficiency.keys(), key=lambda x: feature_efficiency[x])
        print(f"• MOST EFFICIENT: {most_efficient} features (R² gain per feature: {feature_efficiency[most_efficient]:.6f})")

# Diminishing returns analysis
r2_values = [feature_impact_results[k]['r2'] for k in sorted(feature_impact_results.keys())]
if len(r2_values) > 2:
    # Calculate second derivative to detect diminishing returns
    first_diffs = [r2_values[i+1] - r2_values[i] for i in range(len(r2_values)-1)]
    second_diffs = [first_diffs[i+1] - first_diffs[i] for i in range(len(first_diffs)-1)]

    diminishing_point = None
    for i, second_diff in enumerate(second_diffs):
        if second_diff < -0.001:  # Significant decrease in improvement rate
            diminishing_point = feature_counts[i+2]
            break

    if diminishing_point:
        print(f"• DIMINISHING RETURNS START: Around {diminishing_point} features")
    else:
        print(f"• DIMINISHING RETURNS: Not clearly detected in tested range")

# Visualization
plt.figure(figsize=(15, 10))

# Plot 1: R² vs number of features
plt.subplot(2, 3, 1)
k_values = sorted(feature_impact_results.keys())
r2_values = [feature_impact_results[k]['r2'] for k in k_values]
plt.plot(k_values, r2_values, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Features')
plt.ylabel('R² Score')
plt.title('Model Performance vs Feature Count')
plt.grid(True, alpha=0.3)

# Plot 2: RMSE vs number of features
plt.subplot(2, 3, 2)
rmse_values = [feature_impact_results[k]['rmse'] for k in k_values]
plt.plot(k_values, rmse_values, 'ro-', linewidth=2, markersize=8)
plt.xlabel('Number of Features')
plt.ylabel('RMSE')
plt.title('Prediction Error vs Feature Count')
plt.grid(True, alpha=0.3)

# Plot 3: Training time vs number of features
plt.subplot(2, 3, 3)
time_values = [feature_impact_results[k]['time'] for k in k_values]
plt.plot(k_values, time_values, 'go-', linewidth=2, markersize=8)
plt.xlabel('Number of Features')
plt.ylabel('Training Time (seconds)')
plt.title('Computational Cost vs Feature Count')
plt.grid(True, alpha=0.3)

# Plot 4: Feature efficiency (R² gain per added feature)
plt.subplot(2, 3, 4)
if feature_efficiency:
    eff_k = sorted(feature_efficiency.keys())
    eff_values = [feature_efficiency[k] for k in eff_k]
    plt.bar(eff_k, eff_values, alpha=0.7)
    plt.xlabel('Number of Features')
    plt.ylabel('R² Gain per Added Feature')
    plt.title('Feature Addition Efficiency')
    plt.grid(True, alpha=0.3)

# Plot 5: Marginal improvement
plt.subplot(2, 3, 5)
if len(r2_values) > 1:
    marginal_improvements = [r2_values[i+1] - r2_values[i] for i in range(len(r2_values)-1)]
    plt.bar(k_values[1:], marginal_improvements, alpha=0.7)
    plt.xlabel('Number of Features')
    plt.ylabel('Marginal R² Improvement')
    plt.title('Marginal Benefit of Adding Features')
    plt.grid(True, alpha=0.3)
    plt.xticks(k_values[1:], [str(s) for s in k_values[1:]], rotation=45)

# Plot 6: Performance vs computational cost
plt.subplot(2, 3, 6)
plt.scatter(time_values, r2_values, s=100, alpha=0.7, c=k_values, cmap='viridis')
plt.colorbar(label='Number of Features')
plt.xlabel('Training Time (seconds)')
plt.ylabel('R² Score')
plt.title('Performance vs Computational Trade-off')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 FEATURE ADDITION CONCLUSIONS:")
if len(feature_impact_results) > 1:
    best_r2_k = max(feature_impact_results.keys(), key=lambda x: feature_impact_results[x]['r2'])
    print(f"• BEST PERFORMANCE: {best_r2_k} features (R² = {feature_impact_results[best_r2_k]['r2']:.4f})")

    # Calculate improvement from minimum to maximum features
    min_k = min(feature_impact_results.keys())
    max_k = max(feature_impact_results.keys())
    total_improvement = feature_impact_results[max_k]['r2'] - feature_impact_results[min_k]['r2']
    print(f"• TOTAL IMPROVEMENT: +{total_improvement:.4f} R² (from {min_k} to {max_k} features)")

    # Efficiency analysis
    if total_improvement > 0:
        avg_efficiency = total_improvement / (max_k - min_k)
        print(f"• AVERAGE EFFICIENCY: {avg_efficiency:.6f} R² per feature")

print(f"\n⚠️ TRADE-OFFS OBSERVED:")
print(f"• More features generally improve R² but with diminishing returns")
print(f"• Computational cost grows approximately linearly with feature count")
print(f"• Risk of overfitting increases with more features on limited data")
print(f"• Optimal feature count balances performance and complexity")


## Step 60: Music Dataset - Subsampling Impact Analysis


In [ ]:
"""
STEP 60: SUBSAMPLING IMPACT ANALYSIS
===================================
REQUIREMENT ALIGNMENT: Addresses requirement 10 - How subsampling impacts metrics
"""

print("=" * 80)
print("🎯 REQUIREMENT 10: IMPACT OF SUBSAMPLING ON METRICS")
print("=" * 80)

# Progressive subsampling analysis
sample_sizes = [500, 1000, 2000, 3000, 4000, 5000] if len(music_sample) >= 5000 else [100, 250, 500, 1000, 2000]
subsampling_results = {}

print("📊 PROGRESSIVE SUBSAMPLING ANALYSIS:")
print(f"{'Sample Size':<12} {'R²':<10} {'RMSE':<10} {'Training Time':<15} {'Stability':<12}")
print("─" * 75)

for size in sample_sizes:
    if size > len(music_sample):
        continue

    # Multiple runs for stability analysis
    results_runs = []

    for run in range(3):  # 3 runs for stability
        # Sample data
        subsample = music_sample.sample(n=size, random_state=STUDENT_SEED + run)

        # Feature selection on subsample
        X_sub = subsample[feature_cols]
        y_sub = subsample['year']

        selector_sub = SelectKBest(score_func=f_regression, k=min(30, size//20))
        X_sub_selected = selector_sub.fit_transform(X_sub, y_sub)

        # Train-test split
        X_train_sub, X_test_sub, y_train_sub, y_test_sub = train_test_split(
            X_sub_selected, y_sub, test_size=0.2, random_state=STUDENT_SEED + run
        )

        # Scale features
        scaler_sub = StandardScaler()
        X_train_scaled_sub = scaler_sub.fit_transform(X_train_sub)
        X_test_scaled_sub = scaler_sub.transform(X_test_sub)

        # Train model
        import time
        start_time = time.time()
        model_sub = LinearRegression()
        result = evaluate_model(
            model_sub, X_train_scaled_sub, X_test_scaled_sub,
            y_train_sub, y_test_sub, f"Sample_{size}"
        )
        training_time = time.time() - start_time

        # Store results with correct key names
        results_runs.append({
            'r2': result['r2'],
            'rmse': result['rmse'],  # Make sure this matches the evaluate_model output
            'time': training_time
        })

    # Calculate average and stability metrics
    avg_r2 = np.mean([r['r2'] for r in results_runs])
    avg_rmse = np.mean([r['rmse'] for r in results_runs])
    avg_time = np.mean([r['time'] for r in results_runs])
    r2_std = np.std([r['r2'] for r in results_runs])

    subsampling_results[size] = {
        'r2': avg_r2,
        'rmse': avg_rmse,
        'time': avg_time,
        'stability': r2_std
    }

    print(f"{size:<12} {avg_r2:<10.4f} {avg_rmse:<10.4f} {avg_time:<15.4f} {r2_std:<12.4f}")

## Step 61: Music Dataset - High-Dimensional Polynomial Challenges


In [ ]:
"""
STEP 61: HIGH-DIMENSIONAL POLYNOMIAL REGRESSION CHALLENGES
==========================================================
REQUIREMENT ALIGNMENT: Addresses requirement 11 - Why polynomial regression struggles with many features
"""

print("=" * 80)
print("🎯 REQUIREMENT 11: POLYNOMIAL REGRESSION CHALLENGES WITH HIGH DIMENSIONS")
print("=" * 80)

# Demonstrate polynomial feature explosion
print("📊 FEATURE EXPLOSION DEMONSTRATION:")

original_features = [5, 10, 20, 30]
degrees = [1, 2, 3]

print(f"{'Original Features':<18} {'Degree':<8} {'Polynomial Features':<20} {'Memory (MB)':<12} {'Status':<15}")
print("─" * 80)

for n_features in original_features:
    for degree in degrees:
        # Calculate theoretical polynomial features
        from math import comb

        # Total polynomial features = sum(C(n+d-1, d) for d in 1 to degree)
        if degree == 1:
            poly_features = n_features
        else:
            poly_features = sum(comb(n_features + d - 1, d) for d in range(1, degree + 1))

        # Estimate memory usage (assuming float64 = 8 bytes)
        # Memory = samples × features × 8 bytes
        sample_size = 10000
        memory_bytes = sample_size * poly_features * 8
        memory_mb = memory_bytes / (1024 * 1024)

        # Determine feasibility
        if memory_mb > 1000:
            status = "❌ Too Large"
        elif memory_mb > 100:
            status = "⚠️ Challenging"
        else:
            status = "✅ Feasible"

        print(f"{n_features:<18} {degree:<8} {poly_features:<20} {memory_mb:<12.1f} {status:<15}")

# Practical demonstration with actual feature creation
print(f"\n🔬 PRACTICAL POLYNOMIAL EXPANSION TEST:")

test_features = [5, 10, 15]
expansion_results = {}

for n_feat in test_features:
    print(f"\nTesting {n_feat} features:")

    # Use subset of actual music data
    X_test_subset = music_sample[selected_features[:n_feat]].head(1000)

    for degree in [1, 2, 3]:
        try:
            import time
            start_time = time.time()

            # Create polynomial features
            poly_transformer = PolynomialFeatures(degree=degree, include_bias=False)
            X_poly = poly_transformer.fit_transform(X_test_subset)

            expansion_time = time.time() - start_time

            # Get actual feature count and memory usage
            actual_features = X_poly.shape[1]
            memory_usage = X_poly.nbytes / (1024 * 1024)  # MB

            expansion_results[f"{n_feat}_{degree}"] = {
                'original': n_feat,
                'degree': degree,
                'expanded': actual_features,
                'time': expansion_time,
                'memory': memory_usage
            }

            print(f"  Degree {degree}: {n_feat} → {actual_features} features ({expansion_time:.3f}s, {memory_usage:.1f}MB)")

        except MemoryError:
            print(f"  Degree {degree}: ❌ Memory Error!")
            expansion_results[f"{n_feat}_{degree}"] = {
                'original': n_feat, 'degree': degree, 'expanded': -1,
                'time': -1, 'memory': -1
            }
        except Exception as e:
            print(f"  Degree {degree}: ❌ Error: {str(e)[:50]}...")

# Analyze the curse of dimensionality
print(f"\n📈 CURSE OF DIMENSIONALITY ANALYSIS:")

print(f"\n1. FEATURE EXPLOSION:")
print(f"   • Linear (degree 1): n features")
print(f"   • Quadratic (degree 2): ≈ n²/2 features")
print(f"   • Cubic (degree 3): ≈ n³/6 features")
print(f"   • With 90 audio features:")
print(f"     - Degree 2: ~4,000 features")
print(f"     - Degree 3: ~120,000 features")

print(f"\n2. COMPUTATIONAL COMPLEXITY:")
valid_results = [r for r in expansion_results.values() if r['expanded'] > 0]
if valid_results:
    for result in valid_results:
        ratio = result['expanded'] / result['original']
        print(f"   • {result['original']} → {result['expanded']} features ({ratio:.1f}x expansion)")

print(f"\n3. STATISTICAL CHALLENGES:")
print(f"   • Sample-to-feature ratio becomes unfavorable")
print(f"   • Risk of overfitting increases exponentially")
print(f"   • Model becomes unstable and uninterpretable")
print(f"   • Regularization becomes essential")

# Visualization of polynomial explosion
plt.figure(figsize=(16, 10))

# Plot 1: Feature explosion visualization
plt.subplot(2, 3, 1)
feature_ranges = range(5, 31, 5)
for degree in [1, 2, 3]:
    poly_counts = []
    for n in feature_ranges:
        if degree == 1:
            count = n
        else:
            count = sum(comb(n_features + d - 1, d) for d in range(1, degree + 1))
        poly_counts.append(count)

    plt.plot(feature_ranges, poly_counts, f'o-', label=f'Degree {degree}', linewidth=2, markersize=6)

plt.xlabel('Original Features')
plt.ylabel('Polynomial Features')
plt.title('Feature Explosion by Polynomial Degree')
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)

# Plot 2: Memory requirements
plt.subplot(2, 3, 2)
sample_sizes = [1000, 5000, 10000, 20000]
n_features = 20  # Fixed feature count

for sample_size in sample_sizes:
    memory_requirements = []
    for degree in [1, 2, 3]:
        if degree == 1:
            poly_features = n_features
        else:
            poly_features = sum(comb(n_features + d - 1, d) for d in range(1, degree + 1))

        memory_mb = sample_size * poly_features * 8 / (1024 * 1024)
        memory_requirements.append(memory_mb)

    plt.plot([1, 2, 3], memory_requirements, 'o-', label=f'{sample_size} samples', linewidth=2)

plt.xlabel('Polynomial Degree')
plt.ylabel('Memory Usage (MB)')
plt.title(f'Memory Requirements (20 features)')
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)

# Plot 3: Processing time comparison
plt.subplot(2, 3, 3)
if valid_results:
    processing_data = {}
    for result in valid_results:
        degree = result['degree']
        if degree not in processing_data:
            processing_data[degree] = {'features': [], 'times': []}
        processing_data[degree]['features'].append(result['original'])
        processing_data[degree]['times'].append(result['time'])

    for degree in sorted(processing_data.keys()):
        plt.scatter(processing_data[degree]['features'],
                   processing_data[degree]['times'],
                   label=f'Degree {degree}', s=80)

    plt.xlabel('Original Features')
    plt.ylabel('Processing Time (seconds)')
    plt.title('Polynomial Expansion Time')
    plt.legend()
    plt.grid(True, alpha=0.3)

# Plot 4: Sample-to-feature ratio
plt.subplot(2, 3, 4)
sample_to_feature_ratios = []
feature_counts = []
labels = []

for result in valid_results:
    if result['expanded'] > 0:
        ratio = 10000 / result['expanded']  # Assuming 10k samples
        sample_to_feature_ratios.append(ratio)
        feature_counts.append(result['expanded'])
        labels.append(f"{result['original']}f, d{result['degree']}")

if sample_to_feature_ratios:
    colors = ['green' if r > 10 else 'yellow' if r > 5 else 'red' for r in sample_to_feature_ratios]
    plt.scatter(feature_counts, sample_to_feature_ratios, c=colors, s=100, alpha=0.7)

    for i, label in enumerate(labels):
        plt.annotate(label, (feature_counts[i], sample_to_feature_ratios[i]),
                    xytext=(5, 5), textcoords='offset points', fontsize=8)

    plt.axhline(y=10, color='red', linestyle='--', alpha=0.7, label='Danger Zone')
    plt.xlabel('Polynomial Features')
    plt.ylabel('Sample-to-Feature Ratio')
    plt.title('Statistical Adequacy Check')
    plt.legend()
    plt.grid(True, alpha=0.3)

# Plot 5: Regularization necessity
plt.subplot(2, 3, 5)
degrees = [1, 2, 3]
regularization_strength = [0.001, 0.1, 10]  # Typical alpha values needed

plt.bar(degrees, regularization_strength, alpha=0.7, color=['green', 'yellow', 'red'])
plt.xlabel('Polynomial Degree')
plt.ylabel('Regularization Strength Needed')
plt.title('Regularization Requirements')
plt.yscale('log')
plt.grid(True, alpha=0.3)

# Plot 6: Model interpretability
plt.subplot(2, 3, 6)
interpretability_scores = [90, 30, 5]  # Subjective interpretability percentage
plt.bar(degrees, interpretability_scores, alpha=0.7, color=['green', 'yellow', 'red'])
plt.xlabel('Polynomial Degree')
plt.ylabel('Interpretability Score (%)')
plt.title('Model Interpretability Loss')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 WHY POLYNOMIAL REGRESSION STRUGGLES WITH HIGH DIMENSIONS:")

print(f"\n🔢 MATHEMATICAL CHALLENGES:")
print(f"• FEATURE EXPLOSION: Polynomial terms grow combinatorially")
print(f"• SAMPLE INADEQUACY: Need exponentially more samples")
print(f"• MULTICOLLINEARITY: Polynomial terms highly correlated")
print(f"• NUMERICAL INSTABILITY: Large coefficients, poor conditioning")

print(f"\n💻 COMPUTATIONAL CHALLENGES:")
print(f"• MEMORY EXPLOSION: Storage requirements grow polynomially")
print(f"• PROCESSING TIME: Feature generation becomes bottleneck")
print(f"• MATRIX OPERATIONS: Linear algebra becomes computationally expensive")

print(f"\n📊 STATISTICAL CHALLENGES:")
print(f"• OVERFITTING: Model memorizes noise in high-dimensional space")
print(f"• GENERALIZATION: Poor performance on unseen data")
print(f"• INTERPRETABILITY: Impossible to understand feature interactions")
print(f"• REGULARIZATION DEPENDENCY: Must use strong regularization")

print(f"\n✅ RECOMMENDATIONS FOR HIGH-DIMENSIONAL DATA:")
print(f"• USE DIMENSIONALITY REDUCTION: PCA, feature selection first")
print(f"• APPLY REGULARIZATION: Ridge, Lasso, Elastic Net")
print(f"• CONSIDER ALTERNATIVES: Random Forest, Neural Networks")
print(f"• LIMIT POLYNOMIAL DEGREE: Stay at degree 1-2 for >20 features")
print(f"• USE INTERACTION-ONLY: Avoid pure polynomial powers")



## Step 62: Comprehensive Metrics Display


In [ ]:
"""
## STEP 62: COMPREHENSIVE METRICS DISPLAY
MISSING REQUIREMENT 1: MSE, MAE, RMSE EVALUATION METRICS
========================================================
Your evaluate_model function includes R2 and RMSE but the requirement explicitly asks for MSE, MAE, RMSE display
"""
print("=" * 80)
print("MISSING REQUIREMENT 1: COMPREHENSIVE METRICS DISPLAY")
print("=" * 80)


# Enhanced metrics display for all models
def display_comprehensive_metrics(results_dict, dataset_name="Music"):
    """Display all required metrics: MSE, R2, MAE, RMSE"""
    print(f"\n📊 COMPREHENSIVE METRICS SUMMARY - {dataset_name}:")
    print(f"{'Model':<20} {'R²':<10} {'MSE':<12} {'MAE':<10} {'RMSE':<10}")
    print("─" * 65)

    for model_name, results in results_dict.items():
        r2 = results['r2']
        mse = results['mse']  # This should already be in your evaluate_model
        mae = results['mae']  # This should already be in your evaluate_model
        rmse = results['rmse']

        print(f"{model_name:<20} {r2:<10.4f} {mse:<12.2f} {mae:<10.2f} {rmse:<10.2f}")


# Apply to your existing results
if 'music_results' in globals():
    display_comprehensive_metrics(music_results, "Music Dataset")


## Step 63: Predicted vs Actual Visualization


In [ ]:
"""
## STEP 63: PREDICTED VS ACTUAL VISUALIZATION

MISSING REQUIREMENT 2: PREDICTED VS ACTUAL VISUALIZATION
========================================================
Requirement asks for "Predicted vs actual (subsample plot if too slow)"
"""
print("\n" + "=" * 80)
print("MISSING REQUIREMENT 2: PREDICTED VS ACTUAL PLOTS")
print("=" * 80)


# Create predicted vs actual plots for all models
def plot_predictions_vs_actual(results_dict, y_test, model_names=None, subsample_size=1000):
    """Create predicted vs actual scatter plots"""
    if model_names is None:
        model_names = list(results_dict.keys())

    n_models = len(model_names)
    fig, axes = plt.subplots(1, min(n_models, 3), figsize=(15, 5))  # Max 3 plots per row
    if n_models == 1:
        axes = [axes]

    for i, model_name in enumerate(model_names[:3]):  # Limit to 3 for space
        if model_name in results_dict:
            predictions = results_dict[model_name]['predictions']

            # Subsample for visualization if data is large
            if len(predictions) > subsample_size:
                indices = np.random.choice(len(predictions), subsample_size, replace=False)
                y_plot = y_test.iloc[indices] if hasattr(y_test, 'iloc') else y_test[indices]
                pred_plot = predictions[indices]
            else:
                y_plot = y_test
                pred_plot = predictions

            # Create scatter plot
            axes[i].scatter(y_plot, pred_plot, alpha=0.6, s=20)

            # Add perfect prediction line
            min_val = min(min(y_plot), min(pred_plot))
            max_val = max(max(y_plot), max(pred_plot))
            axes[i].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)

            axes[i].set_xlabel('Actual Year')
            axes[i].set_ylabel('Predicted Year')
            axes[i].set_title(f'{model_name}\nR² = {results_dict[model_name]["r2"]:.4f}')
            axes[i].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# Apply to your music results
if 'music_results' in globals() and 'y_music_test' in globals():
    plot_predictions_vs_actual(music_results, y_music_test)


## Step 64: Error Distribution by Decade Analysis


In [ ]:
"""
## STEP 64: ERROR DISTRIBUTION BY DECADE ANALYSIS

MISSING REQUIREMENT 3: ERROR DISTRIBUTION BY DECADE
===================================================
Requirement asks to "Report or visualize error distribution by decade"
"""
print("\n" + "=" * 80)
print("MISSING REQUIREMENT 3: ERROR DISTRIBUTION BY DECADE")
print("=" * 80)


def analyze_error_by_decade(results_dict, y_test, test_indices=None):
    """Analyze prediction errors by decade"""

    # Get the test data with decades
    if test_indices is not None:
        test_data = music_sample.iloc[test_indices].copy()
    else:
        # If no indices provided, recreate the test set
        # This is a limitation - ideally we'd store test indices
        print("⚠️ Test indices not available - using approximation")
        test_data = music_sample.sample(len(y_test), random_state=STUDENT_SEED)

    test_data['decade'] = (test_data['year'] // 10) * 10

    # For each model, calculate errors by decade
    for model_name, results in results_dict.items():
        predictions = results['predictions']
        errors = predictions - y_test.values if hasattr(y_test, 'values') else predictions - y_test

        # Add errors to test data
        test_data[f'{model_name}_error'] = errors
        test_data[f'{model_name}_abs_error'] = np.abs(errors)

    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Plot 1: Average error by decade
    decade_stats = test_data.groupby('decade').agg({
        f'{list(results_dict.keys())[0]}_error': ['mean', 'std', 'count']
    }).round(3)

    decades = sorted(test_data['decade'].unique())
    for i, model_name in enumerate(list(results_dict.keys())[:2]):  # Limit to 2 models
        if f'{model_name}_error' in test_data.columns:
            decade_errors = test_data.groupby('decade')[f'{model_name}_error'].mean()
            axes[0, i].bar(decades, decade_errors, alpha=0.7)
            axes[0, i].set_xlabel('Decade')
            axes[0, i].set_ylabel('Average Prediction Error (years)')
            axes[0, i].set_title(f'{model_name}: Error by Decade')
            axes[0, i].grid(True, alpha=0.3)
            axes[0, i].axhline(y=0, color='red', linestyle='--')

    # Plot 2: Error distribution violin plot
    if len(results_dict) > 0:
        model_name = list(results_dict.keys())[0]
        if f'{model_name}_error' in test_data.columns:
            decade_labels = [f"{int(d)}s" for d in decades]
            error_data = [test_data[test_data['decade'] == d][f'{model_name}_error'].values
                          for d in decades]

            axes[1, 0].violinplot(error_data, positions=range(len(decades)))
            axes[1, 0].set_xticks(range(len(decades)))
            axes[1, 0].set_xticklabels(decade_labels, rotation=45)
            axes[1, 0].set_ylabel('Prediction Error (years)')
            axes[1, 0].set_title(f'{model_name}: Error Distribution by Decade')
            axes[1, 0].grid(True, alpha=0.3)
            axes[1, 0].axhline(y=0, color='red', linestyle='--')

    # Plot 3: Absolute error by decade
    if len(results_dict) > 0:
        model_name = list(results_dict.keys())[0]
        if f'{model_name}_abs_error' in test_data.columns:
            decade_abs_errors = test_data.groupby('decade')[f'{model_name}_abs_error'].mean()
            axes[1, 1].plot(decades, decade_abs_errors, 'o-', linewidth=2, markersize=8)
            axes[1, 1].set_xlabel('Decade')
            axes[1, 1].set_ylabel('Average Absolute Error (years)')
            axes[1, 1].set_title(f'{model_name}: Absolute Error Trend')
            axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print statistical summary
    print(f"\n📊 ERROR BY DECADE SUMMARY:")
    for decade in sorted(decades):
        decade_data = test_data[test_data['decade'] == decade]
        count = len(decade_data)

        print(f"\n{int(decade)}s ({count} songs):")
        for model_name in results_dict.keys():
            if f'{model_name}_error' in test_data.columns:
                mean_error = decade_data[f'{model_name}_error'].mean()
                std_error = decade_data[f'{model_name}_error'].std()
                mae_error = decade_data[f'{model_name}_abs_error'].mean()

                print(f"  {model_name}: Mean Error = {mean_error:6.2f}±{std_error:.2f}, MAE = {mae_error:.2f}")


# Apply to your music results
if 'music_results' in globals() and 'y_music_test' in globals():
    analyze_error_by_decade(music_results, y_music_test)


## Step 56: Music Dataset - Enhanced Audio Trends and Evolution Analysis


In [ ]:
"""
MISSING REQUIREMENT 4: AUDIO TRENDS OVER TIME ANALYSIS
======================================================
Requirement asks: "How do timbre features evolve over time? (modern songs higher complexity)"
"""
print("\n" + "=" * 80)
print("MISSING REQUIREMENT 4: ENHANCED AUDIO TRENDS ANALYSIS")
print("=" * 80)


def analyze_audio_evolution_trends(music_data, selected_features):
    """Detailed analysis of how audio features evolved over time"""

    print("🎵 AUDIO FEATURE EVOLUTION ANALYSIS:")

    # Group by decades for trend analysis
    decade_groups = music_data.groupby('decade')

    # Calculate complexity metrics
    complexity_metrics = {}

    for decade, group in decade_groups:
        # Feature variance as complexity measure
        feature_variance = group[selected_features].var().mean()

        # Feature range as diversity measure
        feature_ranges = (group[selected_features].max() - group[selected_features].min()).mean()

        # Spectral complexity (average of first 12 features - timbre averages)
        timbre_avg_features = [f for f in selected_features if int(f.split('_')[1]) <= 12]
        if timbre_avg_features:
            spectral_complexity = group[timbre_avg_features].std().mean()
        else:
            spectral_complexity = 0

        complexity_metrics[decade] = {
            'variance': feature_variance,
            'range': feature_ranges,
            'spectral_complexity': spectral_complexity,
            'song_count': len(group)
        }

    # Create comprehensive visualization
    decades = sorted(complexity_metrics.keys())

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # Plot 1: Feature variance evolution
    variances = [complexity_metrics[d]['variance'] for d in decades]
    axes[0, 0].plot(decades, variances, 'bo-', linewidth=3, markersize=8)
    axes[0, 0].set_xlabel('Decade')
    axes[0, 0].set_ylabel('Average Feature Variance')
    axes[0, 0].set_title('Musical Complexity Evolution\n(Higher = More Complex)')
    axes[0, 0].grid(True, alpha=0.3)

    # Plot 2: Spectral complexity
    spectral_complexity = [complexity_metrics[d]['spectral_complexity'] for d in decades]
    axes[0, 1].plot(decades, spectral_complexity, 'ro-', linewidth=3, markersize=8)
    axes[0, 1].set_xlabel('Decade')
    axes[0, 1].set_ylabel('Timbre Complexity')
    axes[0, 1].set_title('Timbral Characteristics Evolution')
    axes[0, 1].grid(True, alpha=0.3)

    # Plot 3: Feature ranges
    ranges = [complexity_metrics[d]['range'] for d in decades]
    axes[0, 2].plot(decades, ranges, 'go-', linewidth=3, markersize=8)
    axes[0, 2].set_xlabel('Decade')
    axes[0, 2].set_ylabel('Average Feature Range')
    axes[0, 2].set_title('Musical Diversity Evolution')
    axes[0, 2].grid(True, alpha=0.3)

    # Plot 4: Top timbre averages evolution
    axes[1, 0].set_title('Top 3 Timbre Features Evolution')
    timbre_features = [f for f in selected_features if int(f.split('_')[1]) <= 12][:3]
    colors = ['red', 'blue', 'green']

    for i, feature in enumerate(timbre_features):
        decade_means = []
        for decade in decades:
            decade_mean = music_data[music_data['decade'] == decade][feature].mean()
            decade_means.append(decade_mean)

        axes[1, 0].plot(decades, decade_means, f'{colors[i][0]}o-',
                        linewidth=2, label=feature, markersize=6)

    axes[1, 0].set_xlabel('Decade')
    axes[1, 0].set_ylabel('Feature Value')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Plot 5: Technology impact visualization
    axes[1, 1].set_title('Technology Era Impact')

    # Define technology eras
    tech_eras = {
        1960: 'Analog Era',
        1980: 'Digital Recording',
        1990: 'CD Era',
        2000: 'Internet/MP3',
        2010: 'Streaming Era'
    }

    # Show complexity trend with technology annotations
    axes[1, 1].plot(decades, variances, 'ko-', linewidth=3, markersize=8)

    # Add technology era annotations
    for year, era in tech_eras.items():
        if year in decades:
            idx = list(decades).index(year)
            axes[1, 1].annotate(era, (year, variances[idx]),
                                xytext=(10, 10), textcoords='offset points',
                                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7),
                                fontsize=9)

    axes[1, 1].set_xlabel('Decade')
    axes[1, 1].set_ylabel('Complexity Measure')
    axes[1, 1].grid(True, alpha=0.3)

    # Plot 6: Modern vs Classic comparison
    axes[1, 2].set_title('Classic vs Modern Comparison')

    classic_era = music_data[music_data['year'] < 1990]
    modern_era = music_data[music_data['year'] >= 1990]

    classic_complexity = classic_era[selected_features].var().mean()
    modern_complexity = modern_era[selected_features].var().mean()

    axes[1, 2].bar(['Classic\n(<1990)', 'Modern\n(≥1990)'],
                   [classic_complexity, modern_complexity],
                   color=['brown', 'cyan'], alpha=0.7)
    axes[1, 2].set_ylabel('Average Feature Variance')
    axes[1, 2].grid(True, alpha=0.3)

    # Add percentage increase
    pct_increase = (modern_complexity - classic_complexity) / classic_complexity * 100
    axes[1, 2].text(0.5, max(classic_complexity, modern_complexity) * 0.8,
                    f'{pct_increase:+.1f}% change', ha='center', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Print detailed insights
    print(f"\n🎼 AUDIO EVOLUTION INSIGHTS:")
    print(f"{'Decade':<8} {'Complexity':<12} {'Spectral':<12} {'Diversity':<12} {'Songs':<8}")
    print("─" * 60)

    for decade in decades:
        metrics = complexity_metrics[decade]
        print(f"{int(decade)}s    {metrics['variance']:<12.4f} {metrics['spectral_complexity']:<12.4f} "
              f"{metrics['range']:<12.2f} {metrics['song_count']:<8}")

    # Trend analysis
    early_complexity = complexity_metrics[decades[0]]['variance']
    recent_complexity = complexity_metrics[decades[-1]]['variance']
    complexity_change = (recent_complexity - early_complexity) / early_complexity * 100

    print(f"\n💡 KEY TRENDS:")
    print(f"• COMPLEXITY EVOLUTION: {complexity_change:+.1f}% change from {int(decades[0])}s to {int(decades[-1])}s")
    print(f"• MODERN MUSIC: {'More' if complexity_change > 0 else 'Less'} complex than classic era")
    print(
        f"• SPECTRAL TRENDS: {'Increasing' if spectral_complexity[-1] > spectral_complexity[0] else 'Decreasing'} timbral complexity")
    print(f"• DIVERSITY: {'Higher' if ranges[-1] > ranges[0] else 'Lower'} feature diversity in recent decades")


# Apply the analysis
if 'music_sample' in globals() and 'selected_features' in globals():
    analyze_audio_evolution_trends(music_sample, selected_features)
